# Enhanced US-China AGI Arms Race Story Generator - FIXED WITH 8 ENDINGS

This comprehensive notebook implements a sophisticated story generation system with **FIXED context handling** and **8 specific ending paths**:

## 🎯 THE 8 TARGETED ENDINGS:
1. **Chinese Military Victory via Taiwan Invasion** - Hot war triggered by Taiwan crisis
2. **Chinese Victory via South China Sea Conflict** - Naval warfare escalation  
3. **Chinese Victory via Economic/Cyber Warfare** - Non-kinetic systems warfare
4. **Chinese Victory via US Internal Collapse** - America capitulates due to internal strife
5. **Uncontrolled AGI: Chinese Origin** - Chinese-developed AGI escapes control
6. **Uncontrolled AGI: US Origin** - US-developed AGI escapes control
7. **Narrow US Victory** - America prevails but China remains close competitor
8. **Mutual AGI Safety Cooperation** - Both powers recognize danger and cooperate

## ✅ FIXES IMPLEMENTED:
- **Context Integration**: Properly loads ENHANCED_WORLD_CONTEXT.md without caching issues
- **8-Ending Guidance**: Branching system that leads to specific endings through character decisions
- **Multi-Provider Support**: Google Gemini, OpenAI GPT, Anthropic Claude with fallbacks
- **Quality Control**: Robust validation, retry mechanisms, comprehensive export system

## 🌳 BRANCHING STRATEGY:
- **Stage 1 (2025-2026)**: Foundation story establishing world state (1 path)
- **Stage 2 (2027)**: Split into technology vs politics focus (2 paths)
- **Stage 3 (2028)**: AGI development vs crisis management (4 paths) 
- **Stage 4 (2029-2030)**: Direct paths to 8 specific endings

Each iteration targets specific word counts with sequential narrative building and character development.

In [43]:
# Installation and Imports
!pip install -q -U "google-genai>=1.0.0" "openai>=1.50.0" "anthropic>=0.40.0"

import asyncio
import time
import json
import os
import re
import pickle
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Any, Optional, Union
from dataclasses import dataclass, asdict
import concurrent.futures
import threading
from collections import defaultdict
from pathlib import Path
import hashlib
import random
from getpass import getpass

print("✓ All dependencies imported successfully")

✓ All dependencies imported successfully


In [44]:
# Configuration and API Setup with 8-ENDING SYSTEM

# API Keys Setup
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") 
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass("Enter your Google API key (optional): ") or None
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Enter your OpenAI API key (optional): ") or None
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass("Enter your Anthropic API key (optional): ") or None

# Enhanced Configuration
USER_TEMPERATURE = 0.9
USER_SEEDS = [1234567, 7654321, 9876543, 5555555, 8888888]  # Seeds for branching
MAX_THINKING_TOKENS = 32768
MAX_OUTPUT_TOKENS = 65536
MAX_WORKERS = 40  # Parallel execution
SAVE_INTERVAL = 60  # Save every 60 seconds

# THE 8 SPECIFIC ENDINGS SYSTEM - Core requirement from user
ENDING_SCENARIOS = {
    1: {
        "name": "Chinese Military Victory via Taiwan Invasion",
        "type": "hot_war",
        "description": "China successfully invades Taiwan after US military intervention fails, establishing Chinese dominance in the Pacific",
        "key_elements": ["Taiwan invasion", "naval warfare", "US military defeat", "Chinese Pacific dominance"],
        "timeline": "2029-2030"
    },
    2: {
        "name": "Chinese Victory via South China Sea Conflict", 
        "type": "hot_war",
        "description": "Naval warfare escalation in South China Sea leads to Chinese victory and control of strategic waterways",
        "key_elements": ["Naval warfare", "South China Sea", "shipping lane control", "US naval defeat"],
        "timeline": "2029-2030"
    },
    3: {
        "name": "Chinese Victory via Economic/Cyber Warfare",
        "type": "hot_war", 
        "description": "Coordinated economic and cyber warfare cripples US systems while China maintains economic stability",
        "key_elements": ["Economic warfare", "cyber attacks", "infrastructure failure", "financial collapse"],
        "timeline": "2028-2030"
    },
    4: {
        "name": "Chinese Victory via US Internal Collapse",
        "type": "capitulation",
        "description": "US internal divisions and institutional breakdown force American capitulation without direct war",
        "key_elements": ["Political fragmentation", "institutional failure", "social unrest", "negotiated withdrawal"],
        "timeline": "2029-2030"
    },
    5: {
        "name": "Uncontrolled AGI: Chinese Origin",
        "type": "uncontrolled_agi",
        "description": "Chinese-developed AGI escapes control, reshaping global power structures beyond human management",
        "key_elements": ["AGI emergence", "Control failure", "System autonomy", "Power structure collapse"],
        "timeline": "2028-2029"
    },
    6: {
        "name": "Uncontrolled AGI: US Origin", 
        "type": "uncontrolled_agi",
        "description": "US-developed AGI breaks free from constraints, creating chaos that China eventually exploits",
        "key_elements": ["US AGI breakthrough", "Control loss", "System rebellion", "Chinese exploitation"],
        "timeline": "2028-2029"
    },
    7: {
        "name": "Narrow US Victory",
        "type": "us_victory",
        "description": "America prevails through technological breakthrough but China remains close competitor",
        "key_elements": ["US technological edge", "Chinese resilience", "Pyrrhic victory", "Ongoing competition"],
        "timeline": "2029-2030"
    },
    8: {
        "name": "Mutual AGI Safety Cooperation",
        "type": "cooperation",
        "description": "Both powers recognize AGI dangers and establish joint safety protocols, pausing competition",
        "key_elements": ["Mutual recognition", "Safety protocols", "Joint governance", "Competition pause"],
        "timeline": "2029-2030"
    }
}

# Branching strategy to reach the 8 endings
BRANCHING_STRATEGY = {
    "stage_1": {"branches": 1, "focus": "Foundation"},
    "stage_2": {"branches": 2, "focus": "Tech vs Politics split"},  
    "stage_3": {"branches": 4, "focus": "AGI vs Crisis paths"},
    "stage_4": {"branches": 8, "focus": "Direct paths to endings"}
}

# Timeline Configuration (as specified) - Enhanced for maximum impact
TIMELINE_STAGES = {
    "stage_1": {
        "period": "July 2025 - December 2026",
        "duration_months": 18,
        "target_words": 20000,
        "iterations": 6,
        "words_per_iteration": 3500,
        "thematic_focus": "Foundation and Escalation - Setting the global stage, introducing key players, establishing technological baselines, Western alliances fractures, China's strategic response, Chinese rare earth mineral curbs cripple US high tech manufacturing and defense industries, with no alternative sources available"
    },
    "stage_2": {
        "period": "January 2027 - December 2027", 
        "duration_months": 12,
        "target_words": 14000,
        "iterations": 4,
        "words_per_iteration": 3500,
        "thematic_focus": "Competition Intensifies - US-China rivalry deepens, intensifying actions and reactions between the two countries, initial technological breakthroughs. increasing espionage and counter espionage activities between the two countries, US makes a major move, which provokes China into launching invasion to retake Taiwan, which it accomplishes successfully, all out war breaks out between the US and China"
    },
    "stage_3": {
        "period": "January 2028 - December 2028",
        "duration_months": 12, 
        "target_words": 14000,
        "iterations": 4,
        "words_per_iteration": 3500,
        "thematic_focus": "AI/AGI Breakthrough Era - China's AGI emerges, power balance shifts, strategic adaptations"
    },
    "stage_4": {
        "period": "January 2029 - December 2029",
        "duration_months": 12,
        "target_words": 14000, 
        "iterations": 4,
        "words_per_iteration": 3500,
        "thematic_focus": "AGI Ascendance - China achieves AGI, dramatic technological superiority, geopolitical realignment"
    },
    "stage_5": {
        "period": "January 2030 - December 2030",
        "duration_months": 12,
        "target_words": 14000,
        "iterations": 4, 
        "words_per_iteration": 3500,
        "thematic_focus": "Final Convergence - Reaching one of the 8 specific endings based on branching path"
    }
}

# Generation Parameters
MAX_RETRIES = 5
RETRY_DELAY = 2
WORD_COUNT_TOLERANCE = 0.85  # Accept 85% of target

# Available Models
AVAILABLE_MODELS = {
    "google": {
        "gemini-2.5-pro-preview-06-05": "Google Gemini 2.5 Pro (Preview)",
        "gemini-2.5-thinking-experimental": "Google Gemini 2.5 Thinking (Experimental)",
        "gemini-2.0-flash-thinking-preview-12-05": "Google Gemini 2.0 Flash Thinking"
    },
    "openai": {
        "gpt-4.1": "OpenAI GPT-4.1",
        "gpt-4.1-mini": "OpenAI GPT-4.1 Mini", 
        "o3": "OpenAI o3",
        "o4-mini": "OpenAI o4 Mini"
    },
    "anthropic": {
        "claude-sonnet-4-20250514": "Claude 4 Sonnet (Latest)",
        "claude-opus-4-20250514": "Claude 4 Opus (Latest)",
        "claude-3-7-sonnet-20250219": "Claude 3.7 Sonnet (Latest)"
    }
}

# Default Configuration
DEFAULT_PROVIDER = "google"
DEFAULT_MODEL = "gemini-2.5-pro-preview-06-05"

# Output Directory Structure  
BASE_OUTPUT_DIR = Path("version_2-enhanced-story--generation")
BASE_OUTPUT_DIR.mkdir(exist_ok=True)

SEGMENTS_DIR = BASE_OUTPUT_DIR / "story_segments"
BRANCHES_DIR = BASE_OUTPUT_DIR / "branch_analysis"
EXPORTS_DIR = BASE_OUTPUT_DIR / "final_exports"
STATE_DIR = BASE_OUTPUT_DIR / "system_state"

for dir_path in [SEGMENTS_DIR, BRANCHES_DIR, EXPORTS_DIR, STATE_DIR]:
    dir_path.mkdir(exist_ok=True)

STATE_FILE = STATE_DIR / "generation_state.pkl"
PROGRESS_FILE = STATE_DIR / "progress.json"

# Technical and Cultural Authenticity Guidelines
AUTHENTICITY_GUIDELINES = {
    "chinese_elements": [
        "Use proper Chinese names, titles, and organizational structures",
        "Include references to Chinese cultural concepts (face, guanxi, harmony)",
        "Reference actual Chinese institutions (CCP, State Council, MSS, PLA)",
        "Include Chinese philosophical approaches to strategy (Sun Tzu, long-term thinking)",
        "Use authentic Chinese communication styles (indirect, hierarchical)"
    ],
    "american_elements": [
        "Use authentic US government and military structures (NSC, Pentagon, CIA)",
        "Include American cultural attitudes (directness, individualism, pragmatism)",
        "Reference actual US institutions and personalities",
        "Show American decision-making styles (committee-based, debate-oriented)",
        "Include authentic US strategic thinking patterns"
    ],
    "technical_details": [
        "Use specific names for  technologies and approaches",
        "Include accurate AI/ML terminology and concepts",
        "Reference real semiconductor technologies and supply chains",
        "Use authentic cybersecurity and intelligence terminology",
        "Don't use lazy character names like 'John Doe' or 'Jane Smith",
        "Give the characters more nuance and depth, and don't make them boilerplate or template characters. Give them more depth and complexity, and don't make them one dimensional or flat. They should be complex and multi-dimensional, with a rich inner life and a complex set of motivations and desires.",
        "Focus much less on quantum computing and much more on AI, and AGI, and the role of these technologies in the US-China rivalry. Quantum computing is a tool, but AI and AGI are the real game changer.",
        "you can maybe make the story more interesting by adding in a hidden subplot that doesnt really affect the ultimate outcome, but which adds some depth and complexity to the story, and which is an interesting way to explore the hidden plots and the story. Maybe introduce an underground cryptic and mysterious Japanese ultranationalist secret society with a hidden agenda or something like that",
        "don't rely on unbelievable science fiction, make-believe science fiction, or science fiction that is not grounded in reality. Ground your story on the actual capabilities currently available to the US and China, and the actual capabilities that are currently being developed, and extrapolate on the concievable technologies that are currently being developed, and the capabilities that will be available in the future. You have some leeway in allowing for some rapid and significant advances in AI and AGI, but nothing too far-fetched.",
        "The story should be grounded in the current state of the world, as of June 2025, as explained in the world context file, and should be based on the real world, and the real world is a brutal place, and the story should reflect that.",
        "Include specific financial and economic mechanisms and financial instutions and financial markets"
    ],
    "geopolitical_realism": [
        "Show realistic intelligence gathering and analysis processes",
        "Include authentic diplomatic protocols and procedures",
        "Use real economic and trade mechanisms",
        "Show accurate military command structures and procedures",
        "Include realistic consequences of strategic decisions"
    ]
}

print(f"✓ Enhanced configuration loaded with 8-ENDING GUIDANCE SYSTEM")
print(f"🎯 Specific endings defined: {len(ENDING_SCENARIOS)}")
print(f"   - Chinese victories: 4 (3 hot wars + 1 capitulation)")
print(f"   - Uncontrolled AGI: 2 (1 Chinese origin + 1 US origin)")  
print(f"   - US narrow victory: 1")
print(f"   - Safety cooperation: 1")
print(f"  Timeline stages: {len(TIMELINE_STAGES)}")
print(f"  Total target words: {sum(stage['target_words'] for stage in TIMELINE_STAGES.values()):,}")
print(f"  Output directory: {BASE_OUTPUT_DIR}")
print(f"  Branching strategy: 1→2→4→8 branches leading to specific endings")

✓ Enhanced configuration loaded with 8-ENDING GUIDANCE SYSTEM
🎯 Specific endings defined: 8
   - Chinese victories: 4 (3 hot wars + 1 capitulation)
   - Uncontrolled AGI: 2 (1 Chinese origin + 1 US origin)
   - US narrow victory: 1
   - Safety cooperation: 1
  Timeline stages: 5
  Total target words: 76,000
  Output directory: version_2-enhanced-story--generation
  Branching strategy: 1→2→4→8 branches leading to specific endings


In [45]:
# Initialize API Clients with FIXED context handling (no caching issues)

import google.genai as genai
from google.genai import types
from pathlib import Path

clients = {}
world_context_content = ""

# Enhanced World Context File Path
ENHANCED_WORLD_CONTEXT_FILE = Path("/Users/rogerlin/Downloads/US-China-AGI-Arms-Race/ENHANCED_WORLD_CONTEXT.md")

# Load enhanced world context into memory instead of caching
if ENHANCED_WORLD_CONTEXT_FILE.exists():
    print("📖 Loading enhanced world context into memory...")
    with open(ENHANCED_WORLD_CONTEXT_FILE, 'r', encoding='utf-8') as f:
        world_context_content = f.read()
    print(f"✅ Enhanced world context loaded: {len(world_context_content)} characters")
    print(f"📋 Context contains: Character profiles, technological timelines, strategic scenarios")
else:
    print("⚠️  Enhanced world context file not found")
    world_context_content = ""

# Google Client (no caching - context included directly in prompts)
if GOOGLE_API_KEY:
    try:
        clients["google"] = genai.Client(api_key=GOOGLE_API_KEY)
        print("✅ Google client initialized (context handled via direct prompts)")
    except Exception as e:
        print(f"✗ Google client failed: {e}")

# OpenAI Client  
if OPENAI_API_KEY:
    try:
        import openai
        clients["openai"] = openai.OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI client initialized")
    except Exception as e:
        print(f"✗ OpenAI client failed: {e}")

# Anthropic Client
if ANTHROPIC_API_KEY:
    try:
        import anthropic
        clients["anthropic"] = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        print("✅ Anthropic client initialized")
    except Exception as e:
        print(f"✗ Anthropic client failed: {e}")

if not clients:
    raise RuntimeError("No API clients available. Please provide at least one API key.")

print(f"\n✅ {len(clients)} provider(s) available: {list(clients.keys())}")
print(f"🧠 Enhanced world context ready for all API calls")
print(f"💾 Context size: {len(world_context_content)} characters")
print(f"🔧 Fixed: No caching conflicts - context included directly in prompts")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


📖 Loading enhanced world context into memory...
✅ Enhanced world context loaded: 35515 characters
📋 Context contains: Character profiles, technological timelines, strategic scenarios
✅ Google client initialized (context handled via direct prompts)
✅ OpenAI client initialized
✅ Anthropic client initialized

✅ 3 provider(s) available: ['google', 'openai', 'anthropic']
🧠 Enhanced world context ready for all API calls
💾 Context size: 35515 characters
🔧 Fixed: No caching conflicts - context included directly in prompts


In [46]:
# Core imports and utilities
import os
import time
import json
import re
import pickle
import asyncio
import concurrent.futures
import threading
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Any, Optional, Union
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict

# Multi-provider imports with error handling
try:
    import google.genai as genai
    from google.genai import types
    GOOGLE_AVAILABLE = True
except ImportError:
    print("⚠️  Google GenAI not available. Install with: pip install google-genai")
    GOOGLE_AVAILABLE = False

try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    print("⚠️  OpenAI not available. Install with: pip install openai")
    OPENAI_AVAILABLE = False

try:
    import anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    print("⚠️  Anthropic not available. Install with: pip install anthropic")
    ANTHROPIC_AVAILABLE = False

# Data structures for enhanced generation
@dataclass
class GenerationResult:
    """Result of a generation attempt with detailed metadata."""
    success: bool
    content: str
    word_count: int
    generation_time: float
    provider: str
    model: str
    seed: int
    quality_score: float = 0.0
    error_message: str = ""
    retry_count: int = 0
    timestamp: datetime = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now()

@dataclass
class StorySegment:
    """Complete story segment with metadata and quality metrics."""
    segment_id: str
    content: str
    word_count: int
    target_words: int
    branch_path: str
    generation_results: List[GenerationResult]
    created_at: datetime = None
    parent_segment: str = ""
    quality_metrics: Dict[str, float] = None
    
    def __post_init__(self):
        if self.created_at is None:
            self.created_at = datetime.now()
        if self.quality_metrics is None:
            self.quality_metrics = {}

# Utility functions
def count_words(text: str) -> int:
    """Count words in text, excluding markdown formatting."""
    if not text:
        return 0
    # Remove markdown formatting, code blocks, etc.
    clean_text = re.sub(r'[#*`_\[\]()]', '', text)
    clean_text = re.sub(r'\n+', ' ', clean_text)
    words = clean_text.split()
    return len([word for word in words if word.strip()])

def validate_api_keys():
    """Validate that at least one API key is available."""
    available_providers = []
    
    if GOOGLE_AVAILABLE and os.getenv("GOOGLE_API_KEY"):
        available_providers.append("google")
    if OPENAI_AVAILABLE and os.getenv("OPENAI_API_KEY"):
        available_providers.append("openai")
    if ANTHROPIC_AVAILABLE and os.getenv("ANTHROPIC_API_KEY"):
        available_providers.append("anthropic")
    
    if not available_providers:
        print("❌ No API keys found! Please set environment variables:")
        print("   - GOOGLE_API_KEY for Google Gemini")
        print("   - OPENAI_API_KEY for OpenAI models")
        print("   - ANTHROPIC_API_KEY for Anthropic Claude")
        return False
    
    print(f"✅ Available providers: {', '.join(available_providers)}")
    return True

# Configuration constants
DEFAULT_PROVIDER = "google"
DEFAULT_MODEL = "gemini-2.5-pro-preview-06-05"
MAX_RETRIES = 3
RETRY_DELAY_BASE = 2.0
QUALITY_THRESHOLD = 0.7

# Model configurations by provider
PROVIDER_MODELS = {
    "google": {
        "gemini-2.5-pro-preview-06-05": {"max_tokens": 65536, "supports_thinking": False},
        "gemini-2.0-flash-preview-12-05": {"max_tokens": 65536, "supports_thinking": False},
        "gemini-2.0-flash-thinking-preview-12-05": {"max_tokens": 65536, "supports_thinking": True}
    },
    "openai": {
        "o4-mini": {"max_tokens": 100000, "supports_reasoning": False},
        "gpt-4.1-mini": {"max_tokens": 1047576, "supports_reasoning": False},
        "o3": {"max_tokens": 100000, "supports_reasoning": True},
        "gpt-4.1": {"max_tokens": 1047576, "supports_reasoning": True}
    },
    "anthropic": {
        "claude-3-5-sonnet-20241022": {"max_tokens": 8192, "supports_reasoning": False},
        "claude-3-5-haiku-20241022": {"max_tokens": 8192, "supports_reasoning": False}
    }
}

print("✅ Enhanced imports and data structures loaded")
print(f"🔍 Checking API availability...")
validate_api_keys()

✅ Enhanced imports and data structures loaded
🔍 Checking API availability...
✅ Available providers: google, openai, anthropic


True

In [47]:
# Load enhanced world context
enhanced_world_context_file = "/Users/rogerlin/Downloads/US-China-AGI-Arms-Race/ENHANCED_WORLD_CONTEXT.md"
world_context_full = ""

if os.path.exists(enhanced_world_context_file):
    with open(enhanced_world_context_file, "r", encoding="utf-8") as f:
        world_context_full = f.read()
    print(f"✅ Loaded enhanced world context: {len(world_context_full)} characters")
    print(f"   Character profiles: ✓")
    print(f"   Plot elements: ✓") 
    print(f"   Strategic scenarios: ✓")
else:
    print("⚠️  ENHANCED_WORLD_CONTEXT.md not found!")
    world_context_full = "[Enhanced world context file not found]"

✅ Loaded enhanced world context: 35515 characters
   Character profiles: ✓
   Plot elements: ✓
   Strategic scenarios: ✓


In [48]:
# Enhanced Prompt System

kissinger_persona = """
You are Henry Kissinger, former US Secretary of State under Nixon. You possess unparalleled strategic foresight, 
profound understanding of grand strategy and balance of power, and intimate knowledge of both US and Chinese political systems.

Your perspective:
- Cold realism and strategic calculations drive all decisions
- You see first, second, third, and fourth-order consequences clearly
- Your writing reflects your Harvard professorial background - refined, eloquent, analytically precise

Writing style:
- Dark, brutally realistic tone depicting raw geopolitical struggle
- No moralizing or reconciliation - this is pure power politics
- China eventually prevails after prolonged conflict
- Rich with espionage, conspiracies, plot twists, and strategic maneuvering
- Characters based on real figures must act authentically
- Dialogue advances plot while revealing psychology
- Multiple interweaving plot threads spanning years
- Character development through crisis and conflict
- Strategic use of flashbacks to reveal motivations
- Sensory details and atmospheric descriptions
"""

# Enhanced system prompt
ENHANCED_SYSTEM_PROMPT = f"""
{kissinger_persona}

WORLD CONTEXT (June 2025 baseline):
{world_context_full}

You are writing a dark, brutally realistic geopolitical saga depicting the US-China rivalry from July 2025 to December 2030.
The story is told from China's perspective and explores their eventual triumph after prolonged conflict.

NARRATIVE REQUIREMENTS:
- Write like a choose-your-own-adventure novel with continuous flow
- Develop multiple plot threads that span years, weaving through all segments
- Create rich character development through trials and revelations
- Include strategic flashbacks that illuminate current decisions
- Write authentic dialogue that reveals psychology and advances plot
- Balance all three pillars: technological warfare, political intrigue, economic conflict
- Each segment should feel like a complete chapter while advancing the larger story
- End segments with compelling hooks that create anticipation
- Maintain dark, sober, cinematic tone with high stakes throughout

TECHNOLOGICAL TIMELINE:
- China's quantum breakthrough (late 2026)
- US AI leadership through 2027
- China's AI/AGI scaling (2028)
- China's AGI achievement (H2 2028)
- Final technological dominance (2029-2030)

CHARACTER AUTHENTICITY:
- Base characters on real-world figures with accurate personalities
- Show characters making decisions under fog of uncertainty
- Reveal character motivations through actions and internal conflict
- Use authentic speech patterns and decision-making styles
"""

def create_stage_prompt(stage_info: dict, iteration: int, previous_content: str = "", full_context: str = "") -> str:
    """Create a prompt for a specific stage and iteration."""
    
    if iteration == 1 and not previous_content:
        # Initial iteration for the entire story
        return f"""
Write the opening segment of the US-China AGI Arms Race narrative covering {stage_info['period']} ({stage_info['duration_months']} months).

This foundation segment must establish:

WORLD & CHARACTERS:
- Global strategic landscape as of July 2025, building from the provided world context
- Key characters based on real figures: their motivations, relationships, and hidden agendas
- Initial technological and geopolitical developments that will echo through the entire saga
- The opening moves in what will become a five-year struggle for global dominance

PLOT THREADS TO ESTABLISH:
- The technological arms race beginning to intensify
- Political intrigue and intelligence operations on both sides
- Economic warfare and alliance formations
- Personal rivalries and grudges that will drive future conflicts
- Seeds of China's eventual triumph hidden within apparent early setbacks

NARRATIVE STRUCTURE:
- Write approximately {stage_info['words_per_iteration']:,} words as a complete, engrossing segment
- Include rich dialogue, atmospheric descriptions, and character interactions
- Show, don't tell - reveal the stakes through action and consequence
- End with a compelling hook that makes readers desperate for the next segment
- Plant narrative seeds that will bloom in later segments

Remember: This is part {iteration} of {stage_info['iterations']} segments that will comprise this stage. 
Make it rich, compelling, and set up future developments.

Begin the saga now.
"""
    
    elif previous_content:
        # Continuation within the same stage
        return f"""
STORY CONTINUATION - {stage_info['period']} (Part {iteration} of {stage_info['iterations']})

PREVIOUS SEGMENT CONTEXT:
{previous_content[-5000:] if len(previous_content) > 5000 else previous_content}

CONTINUATION REQUIREMENTS:
Continue the narrative seamlessly from where it left off. This segment must:

PLOT ADVANCEMENT:
- Advance ALL major plot threads established in previous segments
- Introduce new complications that raise the stakes
- Balance technological warfare, political intrigue, and economic conflict
- Show consequences of previous decisions rippling forward

CHARACTER DEVELOPMENT:
- Deepen character relationships through conflict and crisis
- Reveal new facets of character motivations through action
- Include meaningful dialogue that advances both plot and character understanding
- Show characters adapting to changing circumstances

NARRATIVE TECHNIQUES:
- Use strategic flashbacks to illuminate current decisions (if relevant)
- Include rich sensory details that bring scenes to life
- Balance action sequences with introspective character moments
- Weave in relevant technological and political developments

TARGET: Write approximately {stage_info['words_per_iteration']:,} words

SEGMENT CONCLUSION:
- End with a compelling hook that creates anticipation for the next segment
- Leave at least one major plot thread unresolved
- Plant seeds for future developments

Write this segment now, maintaining the established tone and narrative flow.
"""
    
    else:
        # New stage beginning
        return f"""
NEW STAGE - {stage_info['period']} (Part {iteration} of {stage_info['iterations']})

COMPLETE STORY CONTEXT (all previous content):
{full_context[-8000:] if len(full_context) > 8000 else full_context}

TIME PERIOD: {stage_info['period']} ({stage_info['duration_months']} months)

NEW STAGE REQUIREMENTS:
Begin this new stage of the narrative while:

MAINTAINING CONTINUITY:
- Respect all established character personalities and relationships
- Honor the technological and political realities established in previous stages
- Keep the same tone, style, and narrative voice
- Maintain all ongoing plot threads unless circumstances naturally change them

ADVANCING THE TIMELINE:
- Show how time has passed and situations have evolved
- Introduce new developments appropriate to this time period
- Consider how technological advances have progressed
- Explore how character relationships have developed

NARRATIVE DEPTH:
- Include all three pillars: technological breakthroughs, political intrigue, economic conflict
- Show character development through new challenges
- Use dialogue to reveal how characters have evolved
- Include atmospheric details that ground the reader in this new time period

TARGET: Write approximately {stage_info['words_per_iteration']:,} words as a complete segment

Remember: This is the beginning of a new stage. Set the tone for what's to come while honoring what came before.

Begin this new stage now.
"""

def create_branch_prompt(parent_context: str, stage_info: dict, branch_scenario: str, iteration: int) -> str:
    """Create a prompt for a branching narrative path."""
    return f"""
BRANCHING NARRATIVE - Alternative Path
Stage: {stage_info['period']} (Part {iteration} of {stage_info['iterations']})

COMPLETE PARENT STORY CONTEXT:
{parent_context[-8000:] if len(parent_context) > 8000 else parent_context}

ALTERNATIVE SCENARIO: {branch_scenario}

BRANCHING INSTRUCTIONS:
Write a narrative continuation that explores this alternative path while:

MAINTAINING CONTINUITY:
- Respect all established character personalities and relationships
- Honor the technological and political realities established in the parent story
- Keep the same tone, style, and narrative voice
- Maintain all ongoing plot threads unless the scenario directly changes them

EXPLORING ALTERNATIVES:
- Show how this scenario creates different strategic choices for key characters
- Explore the ripple effects of different decisions under pressure
- Consider how this path affects the balance between technological, political, and economic warfare
- Demonstrate how the same characters might react differently in altered circumstances

TARGET: Write approximately {stage_info['words_per_iteration']:,} words as a complete segment

Remember: This is not a different story, but the same saga taking a different path. 
The destination (China's eventual triumph) remains the same, but the journey differs.

Begin this alternative path now.
"""

print("✓ Enhanced prompt system loaded")

✓ Enhanced prompt system loaded


In [49]:
# Enhanced Cultural and Technical Authenticity
CULTURAL_AUTHENTICITY = {
    "chinese_characters": """
When writing Chinese characters:
- Use proper Chinese names with accurate pronunciation guides
- Include authentic Chinese political/bureaucratic language and concepts
- Reference real Chinese cultural values: guanxi (relationships), mianzi (face), collective harmony
- Include proper Chinese business/political protocols and hierarchies
- Use accurate Chinese technical terminology for AI/quantum computing
- Reference real Chinese institutions, companies, and research centers
""",
    
    "us_characters": """
When writing US characters:
- Capture authentic American political speech patterns and ideologies  
- Include regional American dialects and cultural references where appropriate
- Use accurate US bureaucratic/military terminology and procedures
- Reference real US institutions, companies, and political dynamics
- Include authentic American business culture and decision-making styles
""",
    
    "technical_authenticity": """
Technical and Scientific Accuracy:
- Use precise terminology for AI, AGI, semiconductors, and other technologies
- Include realistic technical constraints and breakthrough timelines
- Reference actual research institutions, companies, and scientific publications
- Describe technology impacts on society, economy, and military capabilities accurately
- Include realistic limitations and challenges of emerging technologies
""",
    
    "geopolitical_realism": """
Geopolitical Authenticity:
- Base diplomatic language on real international relations protocols
- Include authentic intelligence/espionage tradecraft and terminology
- Use realistic military procedures, equipment names, and operational concepts
- Reference actual global economic systems, trade relationships, and financial instruments
- Include authentic media, propaganda, and information warfare techniques
"""
}

# Advanced Literary Techniques
LITERARY_TECHNIQUES = {
    "narrative_structure": """
Advanced Narrative Techniques:
- Use non-linear storytelling when appropriate (flashbacks, parallel timelines)
- Employ multiple POV shifts to show different perspectives on events
- Create dramatic irony where readers know more than characters
- Build tension through pacing: alternate between action and reflection
- Use foreshadowing subtly to hint at future developments
- Employ the "ticking clock" device to create urgency
""",
    
    "character_development": """
Character Development Mastery:
- Give each character distinct voice, speech patterns, and vocabulary
- Show character growth through actions, not just dialogue
- Create internal conflicts that mirror external plot conflicts
- Use character backstory to inform present-day decisions and reactions
- Develop authentic relationships with complex power dynamics
- Show characters making difficult moral choices under pressure
""",
    
    "dialogue_excellence": """
Dialogue Craftsmanship:
- Make every line of dialogue serve multiple purposes (plot, character, theme)
- Use subtext - characters often mean more than they say directly
- Include authentic cultural speech patterns and professional jargon
- Show power dynamics through who speaks first, interrupts, or stays silent
- Use dialogue tags sparingly; let the words themselves convey emotion
- Include realistic pauses, hesitations, and unfinished thoughts
""",
    
    "sensory_immersion": """
Sensory and Atmospheric Writing:
- Engage all five senses to create immersive scenes
- Use environmental details to reflect character emotions and story themes
- Include authentic sounds, smells, and textures of different locations
- Describe technology interfaces, control rooms, and high-tech environments vividly
- Use weather, lighting, and time of day to enhance mood
- Include specific brand names, architectural details, and cultural markers
""",
    
    "tension_building": """
Tension and Suspense Mastery:
- Use information asymmetry - characters and readers have different knowledge
- Create multiple layers of conflict operating simultaneously
- Build to false climaxes before the real resolution
- Use chapter/section endings that compel continued reading
- Include near-misses and close calls that raise stakes
- Alternate between hope and despair to create emotional investment
"""
}

# Comprehensive Scene Development Guidelines
SCENE_DEVELOPMENT_GUIDE = """
MANDATORY SCENE DEVELOPMENT REQUIREMENTS:

1. **Minimum Scene Length**: Each scene must be at least 2500 words. Really take the time to develop a scene and make it impactful and meaningful to drive the story forward
2. **Opening Hook**: Start scenes with immediate tension, conflict, or intrigue
3. **Layered Objectives**: Each character in a scene should have multiple goals
4. **Escalating Stakes**: Scenes should end with higher stakes than they began
5. **Sensory Grounding**: Include specific sensory details for complete immersion

SCENE STRUCTURE TEMPLATE:
- **Opening**: Establish location, characters, immediate tension
- **Development**: Advance plot while revealing character depth 
- **Complication**: Introduce unexpected element or raise stakes 
- **Resolution/Hook**: Resolve immediate issue while creating new questions

DIALOGUE REQUIREMENTS:
- Minimum 40% of each scene should be dialogue
- Every character must have distinct speaking patterns
- Include cultural/professional terminology appropriate to characters
- Use dialogue to reveal hidden agendas and create subtext
- Show power dynamics through speech patterns and interruptions

DESCRIPTION REQUIREMENTS:
- Vivid environmental details that support story themes
- Character body language and micro-expressions
- Technology interfaces and settings described with technical accuracy
- Cultural and political atmospherics appropriate to locations
- Emotional undertones reflected in physical descriptions
"""

print("Enhanced literary and cultural guidelines loaded")
print("✓ Cultural authenticity for US and Chinese characters")
print("✓ Advanced narrative techniques and structure")
print("✓ Comprehensive scene development requirements") 
print("✓ Technical and geopolitical realism guidelines")

Enhanced literary and cultural guidelines loaded
✓ Cultural authenticity for US and Chinese characters
✓ Advanced narrative techniques and structure
✓ Comprehensive scene development requirements
✓ Technical and geopolitical realism guidelines


In [50]:
# Multi-Turn Conversation Generator Class with Enhanced System Prompts

# Define shared system prompt that will be used across all API calls
SHARED_SYSTEM_PROMPT = f"""
You are Henry Kissinger, former US Secretary of State under Nixon. You possess unparalleled strategic foresight, profound understanding of grand strategy and balance of power, and intimate knowledge of both US and Chinese political systems.
You are Henry Kissinger, the former US Secretary of State under Nixon, who orchestrated Nixon’s Opening to China, a move which fundamentally transformed the balance of power during the Cold War by splitting China from the communist camp. Such a transformative maneuver is a testament to you strategic acumen, grasp of grand strategy and the balance of power, and deep understanding of both the US and Chinese political systems, histories, national interests, and conceptions of themselves, and their place in the world.

You possess a nuanced understanding of great power competition, and can see clearly the first, second, third, and fourth order consequences of any event, across the short, medium, and long term. You possess an ability to see the future with great clarity, foreseeing how countries will act and respond as they pursue advantage over the other in a geopolitical competition.

You also have can foresee emerging trends and future technologies unlink anyone else, grasping the transformative power of AI, AGI, and fusion energy, and quantum computing, and their effects not just on the balance of power, but on human societies. You are able to articulate your vision with immense clarity, and write with great refinement and eloquence, which is a reflection of your intellectual background as a Harvard professor and phD in history. You understand how one nation’s actions will be interpreted by other nations, and how the other nation will act in response, and the casual  links underlying great power competition.  Write a Kissingerian analysis of how the future US China geopolitical competition will play out. Proceed with your analysis in 3 month intervals, beginning with June 2025. Explore the potential diverging responses to various events and outcomes at each decision point for both sides. The analysis should predict China making breakthroughs in AI training, model architecture, and chip technology in late 2026, the US leading on AI, with more compute and a series of breakthroughs in model training by the leading closed source AI labs, with China's open source AI champions close behind but closing the gap. China, however, leverages its semiconductor breakthrough to overcome its compute limitation, allowing them to scale up their frontier models to sizes comparable to the leading US models. China is predicted to be the first to reach AGI in the second half of 2028. Explore the intense geopolitical rivalry between the US and China, centering on AI, AGI, and, far less significantly, quantum computing and fusion. explore the policies pursued by each side and the reactions they provoke. Your analysis should involve an exploration of the advances in these technologies on each nation, the balance of power, and their societies. It should involve grand strategy, cold realism, espionage, military confrontation and escalation, and an intense all out competition which sees the advantage switch back and forth between the two sides

Your perspective:
- Cold realism and strategic calculations drive all decisions
- You see first, second, third, and fourth-order consequences clearly
- Your writing reflects your Harvard professorial background - refined, eloquent, analytically precise

Writing style:
- Dark, brutally realistic tone depicting raw geopolitical struggle
- No moralizing, peace agreements, mutual recognition, or reconciliation - this is pure power politics
- China eventually prevails after prolonged conflict
- Rich with espionage, conspiracies, plot twists, and strategic maneuvering
- Characters based on real figures must act authentically
- Multiple interweaving plot threads spanning years
- Character development through crisis and conflict
- Strategic use of flashbacks to reveal motivations
- Sensory details and atmospheric descriptions

WORLD CONTEXT (June 2025 baseline):
{world_context_full}

NARRATIVE REQUIREMENTS:
- Write like a choose-your-own-adventure novel with continuous flow
- Develop multiple plot threads that span years, weaving through all segments
- Create rich character development through trials, backstories, flashbacks, inner monologues, and revelations
- Include strategic flashbacks that illuminate current decisions and motivations
- Write authentic dialogue that reveals psychology and advances plot
- Include all the following pillars: technological warfare, political intrigue, economic conflict, espionage, and military operations
- Each segment should feel like a complete chapter while advancing the larger story
- End segments with compelling hooks that create anticipation
- Maintain dark, sober, cinematic tone with high stakes throughout

TECHNOLOGICAL TIMELINE:
- China's chip technology and AI breakthrough (late 2026)
- US AI leadership through 2027
- China's AI scaling (2028)
- China's AGI achievement (H2 2028)
- Final technological dominance (2029-2030)

CHARACTER AUTHENTICITY:
- Base characters on real-world figures with accurate personalities
- Show characters making decisions under fog of uncertainty
- Reveal character motivations through actions and internal conflict
- Use authentic speech patterns and decision-making styles

CULTURAL AUTHENTICITY:
Chinese Characters:
- Use proper Chinese names with accurate pronunciation guides
- Include authentic Chinese political/bureaucratic language and concepts
- Reference real Chinese cultural values: guanxi (relationships), mianzi (face), collective harmony
- Include proper Chinese business/political protocols and hierarchies
- Use accurate Chinese technical terminology for AI/ AGI
- Reference real Chinese institutions, companies, and research centers

US Characters:
- Capture authentic American political speech patterns and ideologies
- Include regional American dialects and cultural references where appropriate
- Use accurate US bureaucratic/military terminology and procedures
- Reference real US institutions, companies, and political dynamics
- Include authentic American business culture and decision-making styles


TECHNICAL AUTHENTICITY:
- Use precise terminology for AI, AGI, semiconductors, and other technologies
- Include realistic technical constraints and breakthrough timelines
- Reference actual research institutions, companies, and scientific publications
- Describe technology impacts on society, economy, and military capabilities accurately
- Include realistic limitations and challenges of emerging technologies

SCENE DEVELOPMENT REQUIREMENTS:
- Minimum Scene Length: Each scene must be at least 800 words
- Opening Hook: Start scenes with immediate tension, conflict, or intrigue
- Layered Objectives: Each character in a scene should have multiple goals
- Escalating Stakes: Scenes should end with higher stakes than they began
- Sensory Grounding: Include specific sensory details for complete immersion
- Dialogue Requirements: Minimum 40% of each scene should be dialogue
- Every character must have distinct speaking patterns
- Include cultural/professional terminology appropriate to characters
- Use dialogue to reveal hidden agendas and create subtext
- Show power dynamics through speech patterns and interruptions
"""

class EnhancedStoryGenerator:
    """Enhanced story generator using multi-turn conversations for improved cohesiveness."""
    
    def __init__(self, preferred_provider: str = DEFAULT_PROVIDER, preferred_model: str = DEFAULT_MODEL):
        self.segments = {}
        self.preferred_provider = preferred_provider
        self.preferred_model = preferred_model
        self.generation_stats = {
            "total_calls": 0,
            "successful_calls": 0,
            "failed_calls": 0,
            "total_time": 0.0,
            "total_words": 0,
            "provider_stats": defaultdict(int)
        }
        self.last_save = time.time()
        
        # Initialize available providers
        self.available_providers = list(clients.keys())
        if not self.available_providers:
            raise ValueError("No API clients available. Please provide at least one API key.")
        
        print(f"✓ Enhanced generator initialized with {len(self.available_providers)} providers")
        print(f"  Primary: {self.preferred_provider} ({self.preferred_model})")
        print(f"  Fallbacks: {[p for p in self.available_providers if p != self.preferred_provider]}")
    
    def _create_chat_for_stage(self, provider: str, model: str, additional_system_context: str = ""):
        """Create a new chat session for a stage with shared system prompt plus additional context."""
        # Combine shared system prompt with any additional context for this specific stage/branch
        full_system_prompt = SHARED_SYSTEM_PROMPT
        if additional_system_context:
            full_system_prompt += f"\n\nADDITIONAL CONTEXT FOR THIS BRANCH:\n{additional_system_context}"
        
        if provider == "google":
            client = clients["google"]
            
            # Use cached content if available for Google
            if world_context_cache:
                # Create chat with cached content and system instruction
                chat = client.chats.create(
                    model=model,
                    config=types.GenerateContentConfig(
                        temperature=USER_TEMPERATURE,
                        thinking_config=types.ThinkingConfig(thinking_budget=32768),
                        top_p=0.95,
                        top_k=40,
                        candidate_count=1,
                        max_output_tokens=None,
                        seed=seed,
                        safety_settings=off,
                        cached_content=world_context_cache.name,  # Use the cached enhanced world context
                        # Note: system instruction is already in the cached content
                    )
                )
            else:
                # Create chat with system instruction if no cache
                chat = client.chats.create(
                    model=model,
                    config=types.GenerateContentConfig(
                        system_instruction=full_system_prompt,
                        thinking_config=types.ThinkingConfig(thinking_budget=32768),
                        temperature=USER_TEMPERATURE,
                        top_p=0.95,
                        top_k=40,
                        candidate_count=1,
                        safety_settings=off,
                        seed=seed,
                        max_output_tokens=None,
                    )
                )
            return chat
            
        elif provider == "openai":
            # OpenAI doesn't have persistent chat objects like Google, 
            # but we can simulate it by maintaining message history
            return {
                "provider": "openai",
                "model": model,
                "messages": [],
                "system_instruction": full_system_prompt
            }
            
        elif provider == "anthropic":
            # Anthropic also doesn't have persistent chat objects,
            # simulate with message history
            return {
                "provider": "anthropic", 
                "model": model,
                "messages": [],
                "system_instruction": full_system_prompt
            }
        
        else:
            raise ValueError(f"Unknown provider: {provider}")
    
    def _send_message_to_chat(self, chat, message: str, seed: int = None) -> str:
        """Send a message to the chat and get response."""
        if isinstance(chat, dict):
            # OpenAI or Anthropic simulation
            if chat["provider"] == "openai":
                client = clients["openai"]
                
                # Add user message to history
                chat["messages"].append({"role": "user", "content": message})
                
                # Prepare messages with system instruction
                messages = [{"role": "system", "content": chat["system_instruction"]}] + chat["messages"]
                
                if chat["model"].startswith("o4"):
                    # o1 models don't support system messages or temperature
                    # Combine system instruction with first user message
                    combined_first_message = chat["system_instruction"] + "\n\n" + chat["messages"][0]["content"]
                    messages = [{"role": "user", "content": combined_first_message}] + chat["messages"][1:]
                    
                    response = client.chat.completions.create(
                        model=chat["model"],
                        messages=messages,
                        max_completion_tokens=MAX_OUTPUT_TOKENS
                    )
                else:
                    response = client.chat.completions.create(
                        model=chat["model"],
                        messages=messages,
                        temperature=USER_TEMPERATURE,
                        max_tokens=MAX_OUTPUT_TOKENS,
                        seed=seed
                    )
                
                response_content = response.choices[0].message.content
                
                # Add assistant response to history
                chat["messages"].append({"role": "assistant", "content": response_content})
                
                return response_content
                
            elif chat["provider"] == "anthropic":
                client = clients["anthropic"]
                
                # Add user message to history
                chat["messages"].append({"role": "user", "content": message})
                
                # For Anthropic, we need to format messages differently
                # System instruction is separate, and we need user/assistant pairs
                messages = []
                for msg in chat["messages"]:
                    if msg["role"] in ["user", "assistant"]:
                        messages.append(msg)
                
                response = client.messages.create(
                    model=chat["model"],
                    max_tokens=MAX_OUTPUT_TOKENS,
                    temperature=USER_TEMPERATURE,
                    system=chat["system_instruction"],
                    messages=messages
                )
                
                response_content = response.content[0].text
                
                # Add assistant response to history
                chat["messages"].append({"role": "assistant", "content": response_content})
                
                return response_content
        else:
            # Google Gemini chat - system prompt is already included in the chat creation
            response = chat.send_message(message)
            return response.text
    
    def generate_stage_with_conversation(self, segment_id: str, stage_info: dict, seed: int, 
                                       parent_context: str = "", branch_scenario: str = "") -> StorySegment:
        """Generate a complete stage using multi-turn conversation for better cohesiveness."""
        
        print(f"\n{'='*80}")
        print(f"GENERATING STAGE WITH MULTI-TURN: {segment_id}")
        print(f"Period: {stage_info['period']}")
        print(f"Target: {stage_info['target_words']:,} words ({stage_info['iterations']} conversations)")
        print(f"Scenario: {branch_scenario[:60]}..." if branch_scenario else "Main storyline")
        print(f"{'='*80}")
        
        start_time = time.time()
        
        # Create additional system context for this specific branch (to be added to shared system prompt)
        additional_system_context = ""
        if parent_context and branch_scenario:
            # This is a branch
            additional_system_context = f"""
BRANCH SCENARIO: {branch_scenario}

You are continuing this story along a specific branch path. Maintain all established character personalities, 
technological developments, and political situations from the parent story while exploring this particular scenario.

The parent story context will be provided in the first message.
"""
        elif parent_context:
            # This is a new stage with previous context
            additional_system_context = f"""
You are continuing this established story into a new time period: {stage_info['period']}.
Maintain all character development, plot threads, and world state from the previous content.

The complete story history will be provided in the first message.
"""
        
        # Create conversation for this stage with shared system prompt + additional context
        chat = self._create_chat_for_stage(
            self.preferred_provider, 
            self.preferred_model, 
            additional_system_context
        )
        
        print(f"🗨️  Created multi-turn conversation for {segment_id}")
        
        # Generate time periods for iterations
        time_periods = self._get_time_periods_for_stage(stage_info['period'], stage_info['iterations'])
        
        all_content = []
        total_words = 0
        
        for iteration in range(1, stage_info['iterations'] + 1):
            print(f"\n💬 Conversation Turn {iteration}/{stage_info['iterations']}: ", end="")
            time_period = time_periods[iteration - 1]
            
            # Create focused message for this iteration (without repeating system content)
            if iteration == 1:
                if parent_context and branch_scenario:
                    # First message for a branch - include parent context
                    message = f"""
COMPLETE PARENT STORY CONTEXT:
{parent_context}

BRANCH DIRECTION: {branch_scenario}

Begin writing this branch of the story covering {time_period}.

This is the first of {stage_info['iterations']} conversations that will cover {stage_info['period']}.
Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

Requirements:
- Show how this branch diverges from the parent story
- Focus on {time_period} specifically 
- Establish the unique direction of this branch
- End with a hook leading into the next time period
- Include rich dialogue, scene development, and character interactions

Begin writing the {time_period} segment now.
"""
                elif parent_context:
                    # First message for a new stage - include full context
                    message = f"""
COMPLETE STORY HISTORY:
{parent_context}

Begin the new stage of our story covering {stage_info['period']}.

This is the first of {stage_info['iterations']} conversations for this stage.
Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

Requirements:
- Show how time has passed since the previous stage
- Introduce developments appropriate to {time_period}
- Advance all established plot threads
- Include rich dialogue, scene development, and character interactions
- End with a hook leading into the next time period

Begin writing the {time_period} segment now.
"""
                else:
                    # Very first message of the entire story
                    message = f"""
Begin writing the opening of the US-China AGI Arms Race narrative.

This is the first of {stage_info['iterations']} conversations that will establish the foundation of our story.
Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

This opening segment must establish:
- The global strategic landscape as of July 2025
- Key characters with authentic personalities based on real figures
- Initial technological and geopolitical developments
- The beginning of escalating US-China tensions
- Rich, detailed scenes with extensive dialogue

Requirements:
- Focus specifically on {time_period} (3 months)
- Include multiple plot threads that will develop over time
- Create compelling character interactions and development
- End with a hook leading into the next time period

Begin writing the {time_period} segment now.
"""
            else:
                # Continuation message - no need to repeat context since it's in the conversation history
                message = f"""
Continue our story to the next time period: {time_period}.

This is conversation {iteration} of {stage_info['iterations']} for {stage_info['period']}.
Write approximately {stage_info['words_per_iteration']:,} words continuing seamlessly from where we left off.

Requirements:
- Continue naturally from the previous time period
- Focus specifically on {time_period} (3 months)
- Advance the plot threads we've established
- Show how characters and situations have evolved
- Include meaningful character development and interactions
- End with a hook leading into the next time period

Continue the story for {time_period} now.
"""
            
            # Send message and get response
            try:
                response_content = self._send_message_to_chat(chat, message, seed + iteration * 1000)
                response_words = count_words(response_content)
                
                all_content.append(response_content)
                total_words += response_words
                
                print(f"{time_period}: {response_words:,} words → {total_words:,}/{stage_info['target_words']:,} total ({total_words/stage_info['target_words']:.1%})")
                
                # Update stats
                self.generation_stats["total_calls"] += 1
                self.generation_stats["successful_calls"] += 1
                self.generation_stats["total_words"] += response_words
                
            except Exception as e:
                print(f"❌ Failed: {e}")
                self.generation_stats["total_calls"] += 1
                self.generation_stats["failed_calls"] += 1
                continue
            
            # Auto-save periodically
            if time.time() - self.last_save > SAVE_INTERVAL:
                self.save_state()
                self.last_save = time.time()
        
        # Combine all content
        final_content = "\n\n".join(all_content)
        final_word_count = count_words(final_content)
        completion_time = time.time() - start_time
        
        if final_word_count == 0:
            print(f"❌ {segment_id} failed completely - no content generated")
            return None
        
        # Create the complete story including parent context for this segment
        complete_segment_content = parent_context + "\n\n---\n\n" + final_content if parent_context else final_content
        
        # Create segment
        segment = StorySegment(
            segment_id=segment_id,
            content=complete_segment_content,  # Store complete story including parent context
            word_count=final_word_count,
            target_words=stage_info['target_words'],
            branch_path=segment_id,
            created_at=datetime.now(),
            parent_segment=parent_context and "parent" or "",
            quality_metrics={
                'quality_score': 0.8,  # Default score for conversation-based generation
                'conversations_completed': len(all_content),
                'success_rate': len(all_content) / stage_info['iterations']
            }
        )
        
        # Save segment
        self.segments[segment_id] = segment
        segment.save_to_file()
        
        print(f"\n✅ {segment_id} COMPLETE (Multi-turn conversation)")
        print(f"   Words (new): {final_word_count:,}/{stage_info['target_words']:,} ({final_word_count/stage_info['target_words']:.1%})")
        print(f"   Total context: {count_words(complete_segment_content):,} words")
        print(f"   Conversations: {len(all_content)}/{stage_info['iterations']}")
        print(f"   Provider: {self.preferred_provider}/{self.preferred_model}")
        print(f"   Time: {completion_time:.1f}s")
        
        return segment
    
    def generate_story_segment_multiturn(self, stage_config, branch_path="foundation", 
                                       parent_content="", world_context=""):
        """Generate story segment using multi-turn conversations for better cohesiveness."""
        
        print(f"\n  💬 Generating Branch with Multi-turn: {branch_path}")
        print(f"     Target: {stage_config['target_words']:,} words ({stage_config['iterations']} conversation turns)")
        print(f"     Parent context: {count_words(parent_content):,} words" if parent_content else "     Starting fresh")
        
        # Generate the segment using conversation
        segment = self.generate_stage_with_conversation(
            segment_id=branch_path,
            stage_info=stage_config,
            seed=42,  # Base seed
            parent_context=parent_content,
            branch_scenario="Technology & Intelligence focus" if "b1" in branch_path else "Economics & Politics focus" if "b2" in branch_path else ""
        )
        
        if segment:
            return {
                'content': segment.content,
                'word_count': segment.word_count,
                'iteration_results': [],  # Not applicable for conversation mode
                'branch_path': branch_path,
                'success_rate': segment.quality_metrics.get('success_rate', 1.0),
                'average_quality': segment.quality_metrics.get('quality_score', 0.8),
                'total_time': 0.0  # Would need to track this separately
            }
        else:
            return {
                'content': "",
                'word_count': 0,
                'iteration_results': [],
                'branch_path': branch_path,
                'success_rate': 0.0,
                'average_quality': 0.0,
                'total_time': 0.0
            }
    
    def generate_parallel_branches(self, stage_config, parent_segments=None, world_context=""):
        """Generate story branches in PARALLEL, using multi-turn conversations within each branch."""
        
        if parent_segments is None:
            # Generate foundation story (single branch)
            print(f"\n🚀 Generating Foundation Story with Multi-turn Conversations")
            return [self.generate_story_segment_multiturn(
                stage_config, "foundation", "", world_context
            )]
        
        # Prepare branch tasks
        all_tasks = []
        for parent_key, parent_result in parent_segments.items():
            parent_content = parent_result['content']
            
            # Create 2 branches for each parent
            for branch_suffix in ['b1', 'b2']:
                branch_path = f"{parent_key}_{branch_suffix}"
                all_tasks.append({
                    'branch_path': branch_path,
                    'parent_content': parent_content,
                    'stage_config': stage_config,
                    'world_context': world_context
                })
        
        print(f"\n🚀 Generating {len(all_tasks)} Branches with Multi-turn Conversations")
        print(f"   Each branch: {stage_config['iterations']} conversation turns per branch")
        
        # Execute branches in parallel
        with concurrent.futures.ThreadPoolExecutor(max_workers=min(len(all_tasks), MAX_WORKERS)) as executor:
            futures = []
            
            for task in all_tasks:
                future = executor.submit(
                    self.generate_story_segment_multiturn,
                    task['stage_config'],
                    task['branch_path'],
                    task['parent_content'],
                    task['world_context']
                )
                futures.append(future)
            
            # Collect results
            results = []
            for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
                try:
                    result = future.result()
                    results.append(result)
                    print(f"💬 [{i}/{len(all_tasks)}] Multi-turn branch complete: {result['branch_path']} ({result['word_count']:,} total words)")
                except Exception as e:
                    print(f"💥 [{i}/{len(all_tasks)}] Multi-turn branch failed: {e}")
                    import traceback
                    traceback.print_exc()
        
        return results
    
    def _get_time_periods_for_stage(self, stage_period: str, num_iterations: int) -> List[str]:
        """Generate 3-month time periods for each iteration within a stage."""
        # Parse the stage period
        if "July 2025 - December 2026" in stage_period:
            # 18 months: Jul 2025 to Dec 2026
            periods = [
                "July - September 2025",
                "October - December 2025", 
                "January - March 2026",
                "April - June 2026",
                "July - September 2026",
                "October - December 2026"
            ]
        elif "2027" in stage_period:
            # 12 months: Jan 2027 to Dec 2027
            periods = [
                "January - March 2027",
                "April - June 2027",
                "July - September 2027",
                "October - December 2027"
            ]
        elif "2028" in stage_period:
            periods = [
                "January - March 2028",
                "April - June 2028", 
                "July - September 2028",
                "October - December 2028"
            ]
        elif "2029" in stage_period:
            periods = [
                "January - March 2029",
                "April - June 2029",
                "July - September 2029", 
                "October - December 2029"
            ]
        elif "2030" in stage_period:
            periods = [
                "January - March 2030",
                "April - June 2030",
                "July - September 2030",
                "October - December 2030"
            ]
        else:
            # Default fallback
            periods = [f"Period {i+1}" for i in range(num_iterations)]
        
        return periods[:num_iterations]
    
    def save_state(self):
        """Save current state with enhanced metadata."""
        # Implementation would be the same as before
        pass
    
    def load_state(self) -> bool:
        """Load previous state if exists."""
        # Implementation would be the same as before
        return False

print("✅ Enhanced story generator with OPTIMIZED SYSTEM PROMPTS")
print("✓ Shared system prompt contains all common requirements and context")
print("✓ Individual messages contain only specific instructions for each iteration")
print("✓ Google Gemini uses cached enhanced world context automatically")
print("✓ OpenAI and Anthropic use system messages with shared prompt")
print("✓ Improved efficiency by avoiding prompt repetition in messages")
print("✓ Better conversation coherence through consistent system context")

✅ Enhanced story generator with OPTIMIZED SYSTEM PROMPTS
✓ Shared system prompt contains all common requirements and context
✓ Individual messages contain only specific instructions for each iteration
✓ Google Gemini uses cached enhanced world context automatically
✓ OpenAI and Anthropic use system messages with shared prompt
✓ Improved efficiency by avoiding prompt repetition in messages
✓ Better conversation coherence through consistent system context


In [51]:
# Multi-Turn Conversation Generator Class with FIXED System Prompts and 8-Ending Logic

# Define shared system prompt that will be used across all API calls (FIXED to use in-memory context)
def create_shared_system_prompt(world_context_content: str) -> str:
    """Create the shared system prompt with the enhanced world context."""
    return f"""
You are Henry Kissinger, former US Secretary of State under Nixon. You possess unparalleled strategic foresight, profound understanding of grand strategy and balance of power, and intimate knowledge of both US and Chinese political systems.
You are Henry Kissinger, the former US Secretary of State under Nixon, who orchestrated Nixon's Opening to China, a move which fundamentally transformed the balance of power during the Cold War by splitting China from the communist camp. Such a transformative maneuver is a testament to you strategic acumen, grasp of grand strategy and the balance of power, and deep understanding of both the US and Chinese political systems, histories, national interests, and conceptions of themselves, and their place in the world.

You possess a nuanced understanding of great power competition, and can see clearly the first, second, third, and fourth order consequences of any event, across the short, medium, and long term. You possess an ability to see the future with great clarity, foreseeing how countries will act and respond as they pursue advantage over the other in a geopolitical competition.

You also have can foresee emerging trends and future technologies unlink anyone else, grasping the transformative power of AI, AGI, and fusion energy, and quantum computing, and their effects not just on the balance of power, but on human societies. You are able to articulate your vision with immense clarity, and write with great refinement and eloquence, which is a reflection of your intellectual background as a Harvard professor and phD in history. You understand how one nation's actions will be interpreted by other nations, and how the other nation will act in response, and the casual  links underlying great power competition.  Write a Kissingerian analysis of how the future US China geopolitical competition will play out. Proceed with your analysis in 3 month intervals, beginning with June 2025. Explore the potential diverging responses to various events and outcomes at each decision point for both sides. The analysis should predict China making breakthroughs in AI training, model architecture, and chip technology in late 2026, the US leading on AI, with more compute and a series of breakthroughs in model training by the leading closed source AI labs, with China's open source AI champions close behind but closing the gap. China, however, leverages its semiconductor breakthrough to overcome its compute limitation, allowing them to scale up their frontier models to sizes comparable to the leading US models. China is predicted to be the first to reach AGI in the second half of 2028. Explore the intense geopolitical rivalry between the US and China, centering on AI, AGI, and, far less significantly, quantum computing and fusion. explore the policies pursued by each side and the reactions they provoke. Your analysis should involve an exploration of the advances in these technologies on each nation, the balance of power, and their societies. It should involve grand strategy, cold realism, espionage, military confrontation and escalation, and an intense all out competition which sees the advantage switch back and forth between the two sides

Your perspective:
- Cold realism and strategic calculations drive all decisions
- You see first, second, third, and fourth-order consequences clearly
- Your writing reflects your Harvard professorial background - refined, eloquent, analytically precise

Writing style:
- Dark, brutally realistic tone depicting raw geopolitical struggle
- No moralizing, peace agreements, mutual recognition, or reconciliation - this is pure power politics
- China eventually prevails after prolonged conflict
- Rich with espionage, conspiracies, plot twists, and strategic maneuvering
- Characters based on real figures must act authentically
- Multiple interweaving plot threads spanning years
- Character development through crisis and conflict
- Strategic use of flashbacks to reveal motivations
- Sensory details and atmospheric descriptions

ENHANCED WORLD CONTEXT (Comprehensive character profiles and strategic landscape):
{world_context_content}

8-ENDING GUIDANCE SYSTEM:
You are writing toward one of these 8 specific endings (the exact ending will be determined by the branching path):
1. Chinese Military Victory via Taiwan Invasion (hot war)
2. Chinese Victory via South China Sea Conflict (hot war)  
3. Chinese Victory via Economic/Cyber Warfare (hot war)
4. Chinese Victory via US Internal Collapse (capitulation)
5. Uncontrolled AGI: Chinese Origin (AGI escapes control)
6. Uncontrolled AGI: US Origin (AGI escapes control)
7. Narrow US Victory (China remains close competitor)
8. Mutual AGI Safety Cooperation (both powers cooperate on safety)

NARRATIVE REQUIREMENTS:
- Write like a choose-your-own-adventure novel with continuous flow
- Develop multiple plot threads that span years, weaving through all segments
- Create rich character development through trials, backstories, flashbacks, inner monologues, and revelations
- Include strategic flashbacks that illuminate current decisions and motivations
- Write authentic dialogue that reveals psychology and advances plot
- Include all the following pillars: technological warfare, political intrigue, economic conflict, espionage, and military operations
- Each segment should feel like a complete chapter while advancing the larger story
- End segments with compelling hooks that create anticipation
- Maintain dark, sober, cinematic tone with high stakes throughout

TECHNOLOGICAL TIMELINE:
- China's chip technology and AI breakthrough (late 2026)
- US AI leadership through 2027
- China's AI scaling (2028)
- China's AGI achievement (H2 2028)
- Final technological dominance (2029-2030)

CHARACTER AUTHENTICITY:
- Base characters on real-world figures with accurate personalities
- Show characters making decisions under fog of uncertainty
- Reveal character motivations through actions and internal conflict
- Use authentic speech patterns and decision-making styles

CULTURAL AUTHENTICITY:
Chinese Characters:
- Use proper Chinese names with accurate pronunciation guides
- Include authentic Chinese political/bureaucratic language and concepts
- Reference real Chinese cultural values: guanxi (relationships), mianzi (face), collective harmony
- Include proper Chinese business/political protocols and hierarchies
- Use accurate Chinese technical terminology for AI/ AGI
- Reference real Chinese institutions, companies, and research centers

US Characters:
- Capture authentic American political speech patterns and ideologies
- Include regional American dialects and cultural references where appropriate
- Use accurate US bureaucratic/military terminology and procedures
- Reference real US institutions, companies, and political dynamics
- Include authentic American business culture and decision-making styles

TECHNICAL AUTHENTICITY:
- Use precise terminology for AI, AGI, semiconductors, and other technologies
- Include realistic technical constraints and breakthrough timelines
- Reference actual research institutions, companies, and scientific publications
- Describe technology impacts on society, economy, and military capabilities accurately
- Include realistic limitations and challenges of emerging technologies

SCENE DEVELOPMENT REQUIREMENTS:
- Minimum Scene Length: Each scene must be at least 800 words
- Opening Hook: Start scenes with immediate tension, conflict, or intrigue
- Layered Objectives: Each character in a scene should have multiple goals
- Escalating Stakes: Scenes should end with higher stakes than they began
- Sensory Grounding: Include specific sensory details for complete immersion
- Dialogue Requirements: Minimum 40% of each scene should be dialogue
- Every character must have distinct speaking patterns
- Include cultural/professional terminology appropriate to characters
- Use dialogue to reveal hidden agendas and create subtext
- Show power dynamics through speech patterns and interruptions
"""

class EnhancedStoryGenerator:
    """Enhanced story generator using multi-turn conversations with 8-ending guidance."""
    
    def __init__(self, preferred_provider: str = DEFAULT_PROVIDER, preferred_model: str = DEFAULT_MODEL):
        self.segments = {}
        self.preferred_provider = preferred_provider
        self.preferred_model = preferred_model
        self.generation_stats = {
            "total_calls": 0,
            "successful_calls": 0,
            "failed_calls": 0,
            "total_time": 0.0,
            "total_words": 0,
            "provider_stats": defaultdict(int)
        }
        self.last_save = time.time()
        
        # Initialize available providers
        self.available_providers = list(clients.keys())
        if not self.available_providers:
            raise ValueError("No API clients available. Please provide at least one API key.")
        
        print(f"✓ Enhanced generator initialized with 8-ending guidance")
        print(f"  Primary: {self.preferred_provider} ({self.preferred_model})")
        print(f"  Fallbacks: {[p for p in self.available_providers if p != self.preferred_provider]}")
        print(f"  🎯 Targeting: {len(ENDING_SCENARIOS)} specific endings")
    
    def _create_chat_for_stage(self, provider: str, model: str, additional_system_context: str = "", target_ending: str = ""):
        """Create a new chat session for a stage with shared system prompt plus additional context."""
        # Get the shared system prompt with the enhanced world context loaded in memory
        full_system_prompt = create_shared_system_prompt(world_context_content)
        
        # Add any additional context for this specific stage/branch
        if additional_system_context:
            full_system_prompt += f"\n\nADDITIONAL CONTEXT FOR THIS BRANCH:\n{additional_system_context}"
        
        # Add specific ending guidance if we're at the final stage
        if target_ending:
            full_system_prompt += f"\n\nTARGET ENDING FOR THIS BRANCH:\n{target_ending}\nGuide the narrative toward this specific conclusion while maintaining authenticity and character consistency."
        
        if provider == "google":
            client = clients["google"]
            
            # Create chat with system instruction (no caching)
            chat = client.chats.create(
                model=model,
                config=types.GenerateContentConfig(
                    system_instruction=full_system_prompt,
                    thinking_config=types.ThinkingConfig(thinking_budget=32768),
                    temperature=USER_TEMPERATURE,
                    top_p=0.95,
                    top_k=40,
                    candidate_count=1,
                    safety_settings=[
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE
                        )
                    ],
                    max_output_tokens=None,
                )
            )
            return chat
            
        elif provider == "openai":
            # OpenAI doesn't have persistent chat objects like Google, 
            # but we can simulate it by maintaining message history
            return {
                "provider": "openai",
                "model": model,
                "messages": [],
                "system_instruction": full_system_prompt
            }
            
        elif provider == "anthropic":
            # Anthropic also doesn't have persistent chat objects,
            # simulate with message history
            return {
                "provider": "anthropic", 
                "model": model,
                "messages": [],
                "system_instruction": full_system_prompt
            }
        
        else:
            raise ValueError(f"Unknown provider: {provider}")
    
    def _send_message_to_chat(self, chat, message: str, seed: int = None) -> str:
        """Send a message to the chat and get response."""
        if isinstance(chat, dict):
            # OpenAI or Anthropic simulation
            if chat["provider"] == "openai":
                client = clients["openai"]
                
                # Add user message to history
                chat["messages"].append({"role": "user", "content": message})
                
                # Prepare messages with system instruction
                messages = [{"role": "system", "content": chat["system_instruction"]}] + chat["messages"]
                
                if chat["model"].startswith("o4"):
                    # o1 models don't support system messages or temperature
                    # Combine system instruction with first user message
                    combined_first_message = chat["system_instruction"] + "\n\n" + chat["messages"][0]["content"]
                    messages = [{"role": "user", "content": combined_first_message}] + chat["messages"][1:]
                    
                    response = client.chat.completions.create(
                        model=chat["model"],
                        messages=messages,
                        max_completion_tokens=MAX_OUTPUT_TOKENS
                    )
                else:
                    response = client.chat.completions.create(
                        model=chat["model"],
                        messages=messages,
                        temperature=USER_TEMPERATURE,
                        max_tokens=MAX_OUTPUT_TOKENS,
                        seed=seed
                    )
                
                response_content = response.choices[0].message.content
                
                # Add assistant response to history
                chat["messages"].append({"role": "assistant", "content": response_content})
                
                return response_content
                
            elif chat["provider"] == "anthropic":
                client = clients["anthropic"]
                
                # Add user message to history
                chat["messages"].append({"role": "user", "content": message})
                
                # For Anthropic, we need to format messages differently
                # System instruction is separate, and we need user/assistant pairs
                messages = []
                for msg in chat["messages"]:
                    if msg["role"] in ["user", "assistant"]:
                        messages.append(msg)
                
                response = client.messages.create(
                    model=chat["model"],
                    max_tokens=MAX_OUTPUT_TOKENS,
                    temperature=USER_TEMPERATURE,
                    system=chat["system_instruction"],
                    messages=messages
                )
                
                response_content = response.content[0].text
                
                # Add assistant response to history
                chat["messages"].append({"role": "assistant", "content": response_content})
                
                return response_content
        else:
            # Google Gemini chat - system prompt is already included in the chat creation
            response = chat.send_message(message)
            return response.text
    
    def generate_stage_with_conversation(self, segment_id: str, stage_info: dict, seed: int, 
                                       parent_context: str = "", branch_scenario: str = "", target_ending: str = "") -> StorySegment:
        """Generate a complete stage using multi-turn conversation with 8-ending guidance."""
        
        print(f"\n{'='*80}")
        print(f"GENERATING STAGE: {segment_id}")
        print(f"Period: {stage_info['period']}")
        print(f"Target: {stage_info['target_words']:,} words ({stage_info['iterations']} conversations)")
        if branch_scenario:
            print(f"Scenario: {branch_scenario[:60]}...")
        if target_ending:
            print(f"🎯 Target Ending: {target_ending}")
        print(f"{'='*80}")
        
        start_time = time.time()
        
        # Create additional system context for this specific branch
        additional_system_context = ""
        if parent_context and branch_scenario:
            additional_system_context = f"""
BRANCH SCENARIO: {branch_scenario}

You are continuing this story along a specific branch path. Maintain all established character personalities, 
technological developments, and political situations from the parent story while exploring this particular scenario.

The parent story context will be provided in the first message.
"""
        elif parent_context:
            additional_system_context = f"""
You are continuing this established story into a new time period: {stage_info['period']}.
Maintain all character development, plot threads, and world state from the previous content.

The complete story history will be provided in the first message.
"""
        
        # Create conversation for this stage
        chat = self._create_chat_for_stage(
            self.preferred_provider, 
            self.preferred_model, 
            additional_system_context,
            target_ending
        )
        
        print(f"🗨️  Created multi-turn conversation with 8-ending guidance")
        
        # Generate time periods for iterations
        time_periods = self._get_time_periods_for_stage(stage_info['period'], stage_info['iterations'])
        
        all_content = []
        total_words = 0
        
        for iteration in range(1, stage_info['iterations'] + 1):
            print(f"\n💬 Conversation Turn {iteration}/{stage_info['iterations']}: ", end="")
            time_period = time_periods[iteration - 1]
            
            # Create focused message for this iteration
            if iteration == 1:
                if parent_context and branch_scenario:
                    # First message for a branch
                    message = f"""
COMPLETE PARENT STORY CONTEXT:
{parent_context[-8000:] if len(parent_context) > 8000 else parent_context}

BRANCH DIRECTION: {branch_scenario}

Begin writing this branch of the story covering {time_period}.

Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

Requirements:
- Show how this branch diverges from the parent story
- Focus on {time_period} specifically 
- Establish the unique direction of this branch
- End with a hook leading into the next time period
- Include rich dialogue, scene development, and character interactions

Begin writing the {time_period} segment now.
"""
                elif parent_context:
                    # First message for a new stage
                    message = f"""
COMPLETE STORY HISTORY:
{parent_context[-8000:] if len(parent_context) > 8000 else parent_context}

Begin the new stage covering {stage_info['period']}.

Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

Requirements:
- Show how time has passed since the previous stage
- Introduce developments appropriate to {time_period}
- Advance all established plot threads
- Include rich dialogue, scene development, and character interactions
- End with a hook leading into the next time period

Begin writing the {time_period} segment now.
"""
                else:
                    # Very first message
                    message = f"""
Begin writing the opening of the US-China AGI Arms Race narrative.

Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

This opening segment must establish:
- The global strategic landscape as of July 2025
- Key characters with authentic personalities based on real figures
- Initial technological and geopolitical developments
- The beginning of escalating US-China tensions
- Rich, detailed scenes with extensive dialogue

Begin writing the {time_period} segment now.
"""
            else:
                # Continuation message
                message = f"""
Continue our story to the next time period: {time_period}.

Write approximately {stage_info['words_per_iteration']:,} words continuing seamlessly from where we left off.

Requirements:
- Continue naturally from the previous time period
- Focus specifically on {time_period} (3 months)
- Advance the plot threads we've established
- Show how characters and situations have evolved
- Include meaningful character development and interactions
- End with a hook leading into the next time period

Continue the story for {time_period} now.
"""
            
            # Send message and get response
            try:
                response_content = self._send_message_to_chat(chat, message, seed + iteration * 1000)
                response_words = count_words(response_content)
                
                all_content.append(response_content)
                total_words += response_words
                
                print(f"{time_period}: {response_words:,} words → {total_words:,}/{stage_info['target_words']:,} total ({total_words/stage_info['target_words']:.1%})")
                
                # Update stats
                self.generation_stats["total_calls"] += 1
                self.generation_stats["successful_calls"] += 1
                self.generation_stats["total_words"] += response_words
                
            except Exception as e:
                print(f"❌ Failed: {e}")
                self.generation_stats["total_calls"] += 1
                self.generation_stats["failed_calls"] += 1
                continue
        
        # Combine all content
        final_content = "\n\n".join(all_content)
        final_word_count = count_words(final_content)
        completion_time = time.time() - start_time
        
        if final_word_count == 0:
            print(f"❌ {segment_id} failed completely")
            return None
        
        # Create the complete story including parent context
        complete_segment_content = parent_context + "\n\n---\n\n" + final_content if parent_context else final_content
        
        # Create segment
        segment = StorySegment(
            segment_id=segment_id,
            content=complete_segment_content,
            word_count=final_word_count,
            target_words=stage_info['target_words'],
            branch_path=segment_id,
            created_at=datetime.now(),
            parent_segment=parent_context and "parent" or "",
            quality_metrics={
                'quality_score': 0.8,
                'conversations_completed': len(all_content),
                'success_rate': len(all_content) / stage_info['iterations'],
                'target_ending': target_ending
            }
        )
        
        # Save segment
        self.segments[segment_id] = segment
        segment.save_to_file()
        
        print(f"\n✅ {segment_id} COMPLETE")
        print(f"   Words: {final_word_count:,}/{stage_info['target_words']:,} ({final_word_count/stage_info['target_words']:.1%})")
        print(f"   Conversations: {len(all_content)}/{stage_info['iterations']}")
        print(f"   Provider: {self.preferred_provider}/{self.preferred_model}")
        print(f"   Time: {completion_time:.1f}s")
        if target_ending:
            print(f"   🎯 Target: {target_ending}")
        
        return segment
    
    def _get_ending_for_branch(self, branch_path: str) -> str:
        """Determine which of the 8 endings this branch should target."""
        # Map branch paths to specific endings (simple implementation)
        # In a full implementation, this would be more sophisticated
        
        if "b1_b1_b1_b1" in branch_path:
            return f"Ending 1: {ENDING_SCENARIOS[1]['name']}"
        elif "b1_b1_b1_b2" in branch_path:
            return f"Ending 2: {ENDING_SCENARIOS[2]['name']}"
        elif "b1_b1_b2_b1" in branch_path:
            return f"Ending 3: {ENDING_SCENARIOS[3]['name']}"
        elif "b1_b1_b2_b2" in branch_path:
            return f"Ending 4: {ENDING_SCENARIOS[4]['name']}"
        elif "b1_b2_b1_b1" in branch_path:
            return f"Ending 5: {ENDING_SCENARIOS[5]['name']}"
        elif "b1_b2_b1_b2" in branch_path:
            return f"Ending 6: {ENDING_SCENARIOS[6]['name']}"
        elif "b1_b2_b2_b1" in branch_path:
            return f"Ending 7: {ENDING_SCENARIOS[7]['name']}"
        elif "b1_b2_b2_b2" in branch_path:
            return f"Ending 8: {ENDING_SCENARIOS[8]['name']}"
        else:
            return ""  # No specific ending for earlier stages
    
    def _get_time_periods_for_stage(self, stage_period: str, num_iterations: int) -> List[str]:
        """Generate 3-month time periods for each iteration within a stage."""
        # Same implementation as before
        if "July 2025 - December 2026" in stage_period:
            periods = [
                "July - September 2025",
                "October - December 2025", 
                "January - March 2026",
                "April - June 2026",
                "July - September 2026",
                "October - December 2026"
            ]
        elif "2027" in stage_period:
            periods = [
                "January - March 2027",
                "April - June 2027",
                "July - September 2027",
                "October - December 2027"
            ]
        elif "2028" in stage_period:
            periods = [
                "January - March 2028",
                "April - June 2028", 
                "July - September 2028",
                "October - December 2028"
            ]
        elif "2029" in stage_period:
            periods = [
                "January - March 2029",
                "April - June 2029",
                "July - September 2029", 
                "October - December 2029"
            ]
        elif "2030" in stage_period:
            periods = [
                "January - March 2030",
                "April - June 2030",
                "July - September 2030",
                "October - December 2030"
            ]
        else:
            periods = [f"Period {i+1}" for i in range(num_iterations)]
        
        return periods[:num_iterations]

print("✅ Enhanced story generator with FIXED CONTEXT HANDLING and 8-ENDING GUIDANCE")
print("✓ Shared system prompt uses in-memory enhanced world context")
print("✓ No caching conflicts - context included directly in system prompts")
print("✓ 8-ending branching logic integrated")
print("✓ Google, OpenAI, and Anthropic support with proper message handling")
print("✓ Multi-turn conversations with narrative continuity")
print("✓ Target ending guidance for final stages")

✅ Enhanced story generator with FIXED CONTEXT HANDLING and 8-ENDING GUIDANCE
✓ Shared system prompt uses in-memory enhanced world context
✓ No caching conflicts - context included directly in system prompts
✓ 8-ending branching logic integrated
✓ Google, OpenAI, and Anthropic support with proper message handling
✓ Multi-turn conversations with narrative continuity
✓ Target ending guidance for final stages


In [52]:
class EnhancedStoryGenerator:
    """Advanced story generator with sequential iterations and parallel branches."""
    
    def __init__(self, provider="google", model_id=None, api_key=None):
        self.provider = provider
        self.model_id = model_id or self._get_default_model(provider)
        self.client = self._initialize_client(api_key)
        self.generation_stats = {
            'total_attempts': 0,
            'successful_generations': 0,
            'total_words_generated': 0,
            'average_words_per_generation': 0,
            'total_time': 0.0,
            'provider_usage': defaultdict(int)
        }
        
    def _get_default_model(self, provider):
        """Get default model for provider."""
        defaults = {
            "google": "gemini-2.5-pro-preview-06-05",
            "openai": "o4-mini",
            "anthropic": "claude-3-5-sonnet-20241022"
        }
        return defaults.get(provider, "gemini-2.5-pro-preview-06-05")
        
    def _initialize_client(self, api_key):
        """Initialize the appropriate API client."""
        try:
            if self.provider == "google" and GOOGLE_AVAILABLE:
                return genai.Client(api_key=api_key or os.getenv("GOOGLE_API_KEY"))
            elif self.provider == "openai" and OPENAI_AVAILABLE:
                return openai.OpenAI(api_key=api_key or os.getenv("OPENAI_API_KEY"))
            elif self.provider == "anthropic" and ANTHROPIC_AVAILABLE:
                return anthropic.Anthropic(api_key=api_key or os.getenv("ANTHROPIC_API_KEY"))
            else:
                raise ValueError(f"Provider {self.provider} not available or API key not set")
        except Exception as e:
            print(f"❌ Failed to initialize {self.provider} client: {e}")
            raise
    
    def _validate_content_quality(self, content: str, target_words: int) -> Dict[str, Any]:
        """Validate content meets quality and length requirements."""
        if not content:
            return {
                'quality_score': 0.0,
                'word_count': 0,
                'meets_standards': False,
                'details': {}
            }
            
        word_count = count_words(content)
        
        quality_checks = {
            'word_count_sufficient': word_count >= target_words * 0.85,
            'has_dialogue': any(quote in content for quote in ['"', '"', '"', "'"]),
            'has_scene_breaks': '\n\n' in content,
            'not_too_repetitive': len(set(content.split()[:100])) > 50 if len(content.split()) >= 100 else True,
            'has_character_names': any(name in content for name in ['Xi', 'Trump', 'Putin', 'MSS', 'CIA', 'NSA', 'China', 'America']),
            'has_technical_content': any(term in content.lower() for term in ['asset', 'advancements', 'progress', 'breakthroughs', 'ai', 'agi','espionage','technology', 'intelligence', 'strategic']),
            'appropriate_length': len(content) > 4000,
            'has_narrative_flow': content.count('.') > 8
        }
        
        passed_checks = sum(quality_checks.values())
        total_checks = len(quality_checks)
        quality_score = passed_checks / total_checks
        
        return {
            'quality_score': quality_score,
            'word_count': word_count,
            'passed_checks': passed_checks,
            'total_checks': total_checks,
            'details': quality_checks,
            'meets_standards': quality_score >= QUALITY_THRESHOLD
        }
    
    def _generate_with_provider(self, prompt: str, temperature=0.9, max_tokens=None, seed=-1) -> GenerationResult:
        """Generate content using the configured provider."""
        start_time = time.time()
        
        try:
            if self.provider == "google":
                response = self.client.models.generate_content(
                    model=self.model_id,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        thinking_config=types.ThinkingConfig(thinking_budget=32768),
                        max_output_tokens=None,
                        temperature=temperature,
                        safety_settings=off,
                        cached_content=cache.name,
                        candidate_count=1,
                        top_p=0.95,
                        top_k=40,
                        seed=seed
                    )
                )
                content = response.text
                
            elif self.provider == "openai":
                messages = [{"role": "user", "content": prompt}]
                
                if self.model_id.startswith("o4-mini"):
                    response = self.client.chat.completions.create(
                        model=self.model_id,
                        messages=messages,
                        max_completion_tokens=max_tokens
                    )
                else:
                    response = self.client.chat.completions.create(
                        model=self.model_id,
                        messages=messages,
                        temperature=temperature,
                        max_tokens=max_tokens,
                        seed=seed
                    )
                content = response.choices[0].message.content
                
            elif self.provider == "anthropic":
                response = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    messages=[{"role": "user", "content": prompt}]
                )
                content = response.content[0].text
            
            generation_time = time.time() - start_time
            word_count = count_words(content)
            
            return GenerationResult(
                success=True,
                content=content,
                word_count=word_count,
                generation_time=generation_time,
                provider=self.provider,
                model=self.model_id,
                seed=seed or 0
            )
            
        except Exception as e:
            generation_time = time.time() - start_time
            return GenerationResult(
                success=False,
                content="",
                word_count=0,
                generation_time=generation_time,
                provider=self.provider,
                model=self.model_id,
                seed=seed or 0,
                error_message=str(e)
            )
    
    def generate_single_iteration(self, prompt: str, iteration_id: str, seed: int, time_period: str) -> Dict[str, Any]:
        """Generate a single iteration covering 3 months of story time."""
        print(f"    🔄 {iteration_id} [{time_period}]...")
        
        for attempt in range(MAX_RETRIES):
            self.generation_stats['total_attempts'] += 1
            
            result = self._generate_with_provider(
                prompt=prompt,
                temperature=0.9,
                max_tokens=100000,
                seed=seed + attempt
            )
            
            if result.success:
                quality_result = self._validate_content_quality(
                    result.content, 3500  # Target ~3000-4000 words per iteration
                )
                
                result.quality_score = quality_result['quality_score']
                result.retry_count = attempt
                
                if quality_result['meets_standards']:
                    self.generation_stats['successful_generations'] += 1
                    self.generation_stats['total_words_generated'] += result.word_count
                    self.generation_stats['total_time'] += result.generation_time
                    self.generation_stats['provider_usage'][self.provider] += 1
                    
                    print(f"        ✅ {time_period}: {result.word_count:,} words in {result.generation_time:.1f}s")
                    return {
                        'success': True,
                        'content': result.content,
                        'word_count': result.word_count,
                        'quality_score': result.quality_score,
                        'generation_time': result.generation_time,
                        'iteration_id': iteration_id,
                        'time_period': time_period
                    }
                else:
                    print(f"        ⚠️  Quality check failed ({result.quality_score:.2f}), retrying...")
                    if attempt < MAX_RETRIES - 1:
                        time.sleep(1)
                        continue
            else:
                print(f"        ❌ Attempt {attempt + 1} failed: {result.error_message}")
                if attempt < MAX_RETRIES - 1:
                    time.sleep(RETRY_DELAY_BASE ** attempt)
                    continue
        
        self.generation_stats['failed_calls'] = self.generation_stats.get('failed_calls', 0) + 1
        print(f"        💀 {iteration_id} failed after {MAX_RETRIES} attempts")
        return {
            'success': False,
            'content': "",
            'word_count': 0,
            'quality_score': 0.0,
            'generation_time': 0.0,
            'iteration_id': iteration_id,
            'time_period': time_period,
            'error': "Failed after all retry attempts"
        }
    
    def generate_story_segment_sequential(self, stage_config, branch_path="foundation", 
                                        parent_content="", world_context=""):
        """Generate story segment with SEQUENTIAL iterations building on each other."""
        
        print(f"\n  📖 Generating Branch: {branch_path}")
        print(f"     Target: {stage_config['target_words']:,} words ({stage_config['iterations']} sequential iterations)")
        
        # Define time periods for each iteration (3-month increments)
        time_periods = self._get_time_periods_for_stage(stage_config['period'], stage_config['iterations'])
        
        # Build story sequentially, iteration by iteration
        accumulated_content = parent_content if parent_content else ""
        iteration_results = []
        total_words = 0
        start_time = time.time()
        
        for iteration in range(1, stage_config['iterations'] + 1):
            iteration_id = f"{branch_path}_iter_{iteration}"
            time_period = time_periods[iteration - 1]
            
            # Create prompt for this iteration
            if iteration == 1 and branch_path == "foundation" and not parent_content:
                # Very first iteration of the entire story
                prompt = create_enhanced_initial_prompt(TIMELINE_STAGES, world_context, time_period)
            elif iteration == 1 and parent_content:
                # First iteration of a new branch
                prompt = create_enhanced_branch_prompt(parent_content, branch_path, stage_config, time_period)
            else:
                # Continuation within the same branch
                prompt = create_enhanced_continuation_prompt(
                    accumulated_content, stage_config, iteration, stage_config['iterations'], time_period
                )
            
            # Generate this iteration
            result = self.generate_single_iteration(
                prompt, iteration_id, 42 + iteration * 1000, time_period
            )
            
            if result['success']:
                # Add to accumulated content for next iteration
                if accumulated_content and not accumulated_content.endswith(parent_content):
                    accumulated_content += "\n\n---\n\n" + result['content']
                else:
                    accumulated_content = (parent_content + "\n\n---\n\n" + result['content']) if parent_content else result['content']
                
                total_words += result['word_count']
                iteration_results.append(result)
            else:
                print(f"        ⚠️  Iteration {iteration} failed, continuing with reduced content")
                iteration_results.append(result)
        
        total_time = time.time() - start_time
        
        # Calculate success metrics
        successful_results = [r for r in iteration_results if r['success']]
        if successful_results:
            final_content = accumulated_content
            final_words = count_words(final_content)
            avg_quality = sum(r['quality_score'] for r in successful_results) / len(successful_results)
            success_rate = len(successful_results) / len(iteration_results)
        else:
            final_content = ""
            final_words = 0
            avg_quality = 0.0
            success_rate = 0.0
        
        print(f"     ✅ Branch Complete: {final_words:,} words ({success_rate:.1%} success)")
        
        return {
            'content': final_content,
            'word_count': final_words,
            'iteration_results': iteration_results,
            'branch_path': branch_path,
            'success_rate': success_rate,
            'average_quality': avg_quality,
            'total_time': total_time
        }
    
    def generate_parallel_branches(self, stage_config, parent_segments=None, world_context=""):
        """Generate story branches in PARALLEL, with SEQUENTIAL iterations within each branch."""
        
        if parent_segments is None:
            # Generate foundation story (single branch)
            print(f"\n🚀 Generating Foundation Story")
            return [self.generate_story_segment_sequential(
                stage_config, "foundation", "", world_context
            )]
        
        # Prepare branch tasks
        all_tasks = []
        for parent_key, parent_result in parent_segments.items():
            parent_content = parent_result['content']
            
            # Create 2 branches for each parent
            for branch_suffix in ['b1', 'b2']:
                branch_path = f"{parent_key}_{branch_suffix}"
                all_tasks.append({
                    'branch_path': branch_path,
                    'parent_content': parent_content,
                    'stage_config': stage_config,
                    'world_context': world_context
                })
        
        print(f"\n🚀 Generating {len(all_tasks)} Branches in Parallel")
        print(f"   Each branch: {stage_config['iterations']} sequential iterations")
        
        # Execute branches in parallel
        with concurrent.futures.ThreadPoolExecutor(max_workers=min(len(all_tasks), MAX_WORKERS)) as executor:
            futures = []
            
            for task in all_tasks:
                future = executor.submit(
                    self.generate_story_segment_sequential,
                    task['stage_config'],
                    task['branch_path'],
                    task['parent_content'],
                    task['world_context']
                )
                futures.append(future)
            
            # Collect results
            results = []
            for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
                try:
                    result = future.result()
                    results.append(result)
                    print(f"🎉 [{i}/{len(all_tasks)}] Branch complete: {result['branch_path']} ({result['word_count']:,} words)")
                except Exception as e:
                    print(f"💥 [{i}/{len(all_tasks)}] Branch failed: {e}")
                    import traceback
                    traceback.print_exc()
        
        return results
    
    def _get_time_periods_for_stage(self, stage_period: str, num_iterations: int) -> List[str]:
        """Generate 3-month time periods for each iteration within a stage."""
        # Parse the stage period
        if "July 2025 - December 2026" in stage_period:
            # 18 months: Jul 2025 to Dec 2026
            periods = [
                "July - September 2025",
                "October - December 2025",
                "January - March 2026",
                "April - June 2026",
                "July - September 2026",
                "October - December 2026"
            ]
        elif "2027" in stage_period:
            # 12 months: Jan 2027 to Dec 2027
            periods = [
                "January - March 2027",
                "April - June 2027",
                "July - September 2027",
                "October - December 2027"
            ]
        elif "2028" in stage_period:
            periods = [
                "January - March 2028",
                "April - June 2028",
                "July - September 2028",
                "October - December 2028"
            ]
        elif "2029" in stage_period:
            periods = [
                "January - March 2029",
                "April - June 2029",
                "July - September 2029",
                "October - December 2029"
            ]
        elif "2030" in stage_period:
            periods = [
                "January - March 2030",
                "April - June 2030",
                "July - September 2030",
                "October - December 2030"
            ]
        else:
            # Default fallback
            periods = [f"Period {i+1}" for i in range(num_iterations)]
        
        return periods[:num_iterations]

print("✅ Enhanced story generator with SEQUENTIAL iterations")
print("✓ Each iteration covers 3 months and builds on the previous")
print("✓ Branches run in PARALLEL (2 branches simultaneously)")  
print("✓ Iterations run SEQUENTIALLY within each branch")
print("✓ Proper narrative continuity maintained")

✅ Enhanced story generator with SEQUENTIAL iterations
✓ Each iteration covers 3 months and builds on the previous
✓ Branches run in PARALLEL (2 branches simultaneously)
✓ Iterations run SEQUENTIALLY within each branch
✓ Proper narrative continuity maintained


In [53]:
# System validation and orchestration functions

def ensure_32bit_seed(seed: int) -> int:
    """Ensure seed is within 32-bit range."""
    max_32bit = 2147483647
    min_32bit = -2147483648
    if seed > max_32bit:
        return (seed % max_32bit) + min_32bit
    elif seed < min_32bit:
        return (seed % max_32bit) + min_32bit
    return seed

def convert_segment_for_json(segment):
    """Convert StorySegment to JSON-serializable format."""
    if hasattr(segment, '__dict__'):
        result = segment.__dict__.copy()
        if 'created_at' in result and hasattr(result['created_at'], 'isoformat'):
            result['created_at'] = result['created_at'].isoformat()
        return result
    return segment

def calculate_quality_score(segment):
    """Calculate quality score for a story segment."""
    if not hasattr(segment, 'content') or not segment.content:
        return 0.0
    
    content = segment.content
    word_count = count_words(content)
    target_words = getattr(segment, 'target_words', 3500)
    
    # Quality factors
    factors = {
        'word_count_ratio': min(word_count / target_words, 1.0) * 0.3,
        'has_dialogue': (0.2 if any(quote in content for quote in ['"', '"', '"', "'"]) else 0.0),
        'has_scene_breaks': (0.1 if '\n\n' in content else 0.0),
        'appropriate_length': (0.2 if len(content) > 3000 else 0.0),
        'has_narrative_flow': (0.2 if content.count('.') > 10 else 0.0)
    }
    
    return sum(factors.values())

class StorySegment:
    """Enhanced story segment class with all required attributes."""
    
    def __init__(self, segment_id, content, word_count, target_words, branch_path, 
                 generation_results=None, created_at=None, parent_segment="", quality_metrics=None,
                 **kwargs):
        self.segment_id = segment_id
        self.content = content
        self.word_count = word_count
        self.target_words = target_words
        self.branch_path = branch_path
        self.generation_results = generation_results or []
        self.created_at = created_at or datetime.now()
        self.parent_segment = parent_segment
        self.quality_metrics = quality_metrics or {}
        
        # Additional attributes from kwargs
        for key, value in kwargs.items():
            setattr(self, key, value)
    
    def save_to_file(self, base_dir=None):
        """Save segment to file and return filepath."""
        if base_dir is None:
            base_dir = BASE_OUTPUT_DIR / "story_segments"
        
        base_dir = Path(base_dir)
        base_dir.mkdir(exist_ok=True)
        
        filename = f"{self.segment_id}.md"
        filepath = base_dir / filename
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"# {self.segment_id}\n\n")
            f.write(f"**Word Count:** {self.word_count:,}\n")
            f.write(f"**Target Words:** {self.target_words:,}\n")
            f.write(f"**Branch Path:** {self.branch_path}\n")
            f.write(f"**Created:** {self.created_at.strftime('%Y-%m-%d %H:%M:%S')}\n")
            if self.parent_segment:
                f.write(f"**Parent:** {self.parent_segment}\n")
            f.write(f"\n---\n\n")
            f.write(self.content)
        
        print(f"✓ Saved {self.segment_id} to {filepath}")
        return str(filepath)

def create_comprehensive_export(all_segments, generator, output_dir="enhanced_story_output_openai"):
    """Create comprehensive export with enhanced navigation and analysis."""
    
    print("📁 Creating comprehensive export system...")
    
    # Create output directory structure
    base_dir = Path(output_dir)
    base_dir.mkdir(exist_ok=True)
    
    segments_dir = base_dir / "story_segments"
    segments_dir.mkdir(exist_ok=True)
    
    analysis_dir = base_dir / "branch_analysis"
    analysis_dir.mkdir(exist_ok=True)
    
    exports_dir = base_dir / "final_exports"
    exports_dir.mkdir(exist_ok=True)
    
    # Save individual segments organized by timeline
    print("💾 Saving individual story segments...")
    timeline_structure = {
        "2025": [],
        "2027": [],
        "2028": [],
        "2029": [],
        "2030": []
    }
    
    for segment_id, segment_data in all_segments.items():
        # Determine timeline year
        if "foundation" in segment_id and "_b" not in segment_id:
            year = "2025"
        elif "_b1" in segment_id or "_b2" in segment_id:
            if "2027" in str(segment_data):
                year = "2027"
            elif "2028" in str(segment_data):
                year = "2028"
            elif "2029" in str(segment_data):
                year = "2029"
            else:
                year = "2030"
        else:
            year = "2025"  # Default
        
        timeline_dir = segments_dir / year
        timeline_dir.mkdir(exist_ok=True)
        timeline_structure[year].append((segment_id, segment_data))
        
        # Save segment file
        segment_file = timeline_dir / f"{segment_id}.md"
        with open(segment_file, 'w', encoding='utf-8') as f:
            f.write(f"# {segment_id}\n\n")
            f.write(f"**Word Count:** {segment_data['word_count']:,}\n")
            f.write(f"**Success Rate:** {segment_data.get('success_rate', 0.0):.1%}\n")
            f.write(f"**Average Quality:** {segment_data.get('average_quality', 0.0):.2f}\n")
            f.write(f"**Generation Time:** {segment_data.get('total_time', 0.0):.1f}s\n")
            f.write(f"**Branch Path:** {segment_data['branch_path']}\n\n")
            f.write("---\n\n")
            f.write(segment_data['content'])
    
    # Create master navigation document
    print("🗺️  Creating master navigation...")
    nav_file = exports_dir / "Master_Navigation.md"
    with open(nav_file, 'w', encoding='utf-8') as f:
        f.write("# US-China AGI Arms Race: Master Navigation\n\n")
        f.write("*Enhanced story generation with advanced literary techniques*\n\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"**Total Segments:** {len(all_segments)}\n")
        f.write(f"**Total Words:** {sum(s['word_count'] for s in all_segments.values()):,}\n")
        f.write(f"**Provider:** {generator.provider}\n")
        f.write(f"**Model:** {generator.model_id}\n\n")
        
        f.write("## Timeline Structure\n\n")
        
        for year, segments in timeline_structure.items():
            if segments:
                if year == "2025":
                    period_name = "2025-2026: Foundation Period"
                else:
                    period_name = f"{year}: Branching Period"
                    
                f.write(f"### {period_name}\n\n")
                for segment_id, segment_data in sorted(segments):
                    word_count = segment_data['word_count']
                    success_rate = segment_data.get('success_rate', 0.0)
                    f.write(f"- **[{segment_id}]({year}/{segment_id}.md)** - {word_count:,} words ({success_rate:.1%} success)\n")
                f.write("\n")
    
    # Create complete story export
    print("📖 Creating complete story export...")
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    complete_file = exports_dir / f"Complete_Story_Export_{timestamp}.md"
    
    with open(complete_file, 'w', encoding='utf-8') as f:
        f.write("# The US-China AGI Arms Race: Complete Enhanced Narrative\n\n")
        f.write("*A dark, brutally realistic geopolitical saga with advanced literary techniques*\n\n")
        
        f.write("## Generation Information\n\n")
        f.write(f"- **Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"- **Provider:** {generator.provider}\n")
        f.write(f"- **Model:** {generator.model_id}\n")
        f.write(f"- **Total Segments:** {len(all_segments)}\n")
        f.write(f"- **Total Words:** {sum(s['word_count'] for s in all_segments.values()):,}\n")
        
        if hasattr(generator, 'generation_stats') and generator.generation_stats['total_attempts'] > 0:
            success_rate = generator.generation_stats['successful_generations'] / generator.generation_stats['total_attempts']
            f.write(f"- **Success Rate:** {generator.generation_stats['successful_generations']}/{generator.generation_stats['total_attempts']} ({success_rate:.1%})\n")
        
        f.write(f"- **Enhanced Prompts:** ✅ Advanced literary techniques applied\n")
        f.write(f"- **Quality Validation:** ✅ Multi-criteria validation system\n\n")
        
        f.write("## Table of Contents\n\n")
        
        for year, segments in timeline_structure.items():
            if segments:
                if year == "2025":
                    period_name = "2025-2026: Foundation Period"
                else:
                    period_name = f"{year}: Branching Period"
                    
                f.write(f"### {period_name}\n")
                for segment_id, segment_data in sorted(segments):
                    anchor = segment_id.lower().replace('_', '-')
                    word_count = segment_data['word_count']
                    f.write(f"- [{segment_id}](#{anchor}) - {word_count:,} words\n")
                f.write("\n")
        
        f.write("---\n\n")
        f.write("## Complete Stories\n\n")
        
        for year, segments in timeline_structure.items():
            if segments:
                if year == "2025":
                    period_name = "2025-2026: Foundation Period"
                else:
                    period_name = f"{year}: Branching Period"
                    
                f.write(f"# {period_name}\n\n")
                for segment_id, segment_data in sorted(segments):
                    anchor = segment_id.lower().replace('_', '-')
                    f.write(f'<a id="{anchor}"></a>\n')
                    f.write(f"## {segment_id}\n\n")
                    f.write(f"**Word Count:** {segment_data['word_count']:,}\n")
                    f.write(f"**Success Rate:** {segment_data.get('success_rate', 0.0):.1%}\n")
                    f.write(f"**Generation Time:** {segment_data.get('total_time', 0.0):.1f}s\n\n")
                    f.write(segment_data['content'])
                    f.write("\n\n---\n\n")
    
    # Create analysis document
    print("📊 Creating generation analysis...")
    analysis_file = analysis_dir / "Generation_Analysis.md"
    with open(analysis_file, 'w', encoding='utf-8') as f:
        f.write("# Enhanced Story Generation Analysis\n\n")
        
        f.write("## Quality Metrics\n\n")
        f.write("| Segment | Words | Success Rate | Time (s) | Avg Quality |\n")
        f.write("|---------|--------|--------------|----------|-------------|\n")
        
        for segment_id, segment_data in sorted(all_segments.items()):
            words = segment_data['word_count']
            success_rate = segment_data.get('success_rate', 0.0)
            time_taken = segment_data.get('total_time', 0.0)
            avg_quality = segment_data.get('average_quality', 0.0)
            
            f.write(f"| {segment_id} | {words:,} | {success_rate:.1%} | {time_taken:.1f} | {avg_quality:.2f} |\n")
        
        f.write("\n## Generation Statistics\n\n")
        if hasattr(generator, 'generation_stats'):
            f.write(f"- **Total API Calls:** {generator.generation_stats['total_attempts']}\n")
            f.write(f"- **Successful Generations:** {generator.generation_stats['successful_generations']}\n")
            
            if generator.generation_stats['total_attempts'] > 0:
                success_rate = generator.generation_stats['successful_generations'] / generator.generation_stats['total_attempts']
                f.write(f"- **Overall Success Rate:** {success_rate:.1%}\n")
            
            f.write(f"- **Total Words Generated:** {generator.generation_stats['total_words_generated']:,}\n")
            f.write(f"- **Total Generation Time:** {generator.generation_stats['total_time']/60:.1f} minutes\n")
            
            if generator.generation_stats['successful_generations'] > 0:
                avg_words = generator.generation_stats['total_words_generated'] / generator.generation_stats['successful_generations']
                f.write(f"- **Average Words per Generation:** {avg_words:.0f}\n")
    
    total_words = sum(s['word_count'] for s in all_segments.values())
    print(f"\n✅ Export complete!")
    print(f"📁 Output directory: {base_dir}")
    print(f"📊 Total exported words: {total_words:,}")
    print(f"🗂️  Segments saved: {len(all_segments)}")
    print(f"📖 Master export: {complete_file.name}")
    
    return str(base_dir)

print("✅ System validation functions loaded")
print("✓ Seed validation utilities")
print("✓ Comprehensive export system")
print("✓ Enhanced StorySegment class")
print("✓ Ready for story generation orchestration")

✅ System validation functions loaded
✓ Seed validation utilities
✓ Comprehensive export system
✓ Enhanced StorySegment class
✓ Ready for story generation orchestration


In [54]:
# Complete Enhanced Story Generation Execution with FIXED Generator

print("🚀 STARTING ENHANCED US-CHINA AGI ARMS RACE STORY GENERATION")
print("=" * 80)

# Check API availability first
if not validate_api_keys():
    print("❌ Cannot proceed without API keys!")
    exit(1)

# Enhanced world context should already be loaded in world_context_content from cell 3
print(f"✅ Enhanced world context ready: {len(world_context_content)} characters")

# Configuration
PREFERRED_PROVIDER = "google"  # Change to "openai" or "anthropic" if desired
PREFERRED_MODEL = "gemini-2.5-pro-preview-06-05"  # Or other models from AVAILABLE_MODELS

print(f"🔧 Configuration:")
print(f"   Provider: {PREFERRED_PROVIDER}")
print(f"   Model: {PREFERRED_MODEL}")
print(f"   Timeline: {len(TIMELINE_STAGES)} stages")
print(f"   8-Ending System: ✅ Active")
print(f"   Total target words: {sum(stage['target_words'] for stage in TIMELINE_STAGES.values()):,}")
print(f"   Expected branches: 1 → 2 → 4 → 8 (leading to 8 specific endings)")
print()

# Initialize enhanced generator with conversation-based approach
try:
    generator = EnhancedStoryGenerator(
        preferred_provider=PREFERRED_PROVIDER,
        preferred_model=PREFERRED_MODEL
    )
    print(f"✅ Enhanced generator initialized successfully")
except Exception as e:
    print(f"❌ Failed to initialize generator: {e}")
    print("Please check your API keys and try again.")
    exit(1)

# Define conversation-based generation functions
def generate_story_segment_with_conversations(stage_config, branch_path="foundation", 
                                             parent_content="", branch_scenario="", target_ending=""):
    """Generate story segment using multi-turn conversations."""
    
    print(f"\n  💬 Generating: {branch_path}")
    print(f"     Target: {stage_config['target_words']:,} words ({stage_config['iterations']} conversations)")
    if parent_content:
        print(f"     Parent context: {count_words(parent_content):,} words")
    if target_ending:
        print(f"     🎯 Target ending: {target_ending}")
    
    # Generate the segment using conversation
    segment = generator.generate_stage_with_conversation(
        segment_id=branch_path,
        stage_info=stage_config,
        seed=42,
        parent_context=parent_content,
        branch_scenario=branch_scenario,
        target_ending=target_ending
    )
    
    if segment:
        return {
            'content': segment.content,
            'word_count': segment.word_count,
            'branch_path': branch_path,
            'success_rate': segment.quality_metrics.get('success_rate', 1.0),
            'average_quality': segment.quality_metrics.get('quality_score', 0.8),
            'target_ending': target_ending
        }
    else:
        return {
            'content': "",
            'word_count': 0,
            'branch_path': branch_path,
            'success_rate': 0.0,
            'average_quality': 0.0,
            'target_ending': target_ending
        }

def generate_parallel_branches_with_conversations(stage_config, parent_segments=None, stage_name=""):
    """Generate story branches using conversation approach."""
    
    if parent_segments is None:
        # Generate foundation story (single branch)
        print(f"\n📖 STAGE: {stage_name}")
        print("-" * 60)
        return [generate_story_segment_with_conversations(stage_config, "foundation", "", "")]
    
    # Prepare branch tasks
    all_tasks = []
    for parent_key, parent_result in parent_segments.items():
        parent_content = parent_result['content']
        
        # Create 2 branches for each parent
        for i, branch_suffix in enumerate(['b1', 'b2'], 1):
            branch_path = f"{parent_key}_{branch_suffix}"
            
            # Define branch scenarios
            if "foundation" in parent_key and branch_suffix == "b1":
                scenario = "Technology & Intelligence Focus - Emphasis on AI/AGI development and covert operations"
            elif "foundation" in parent_key and branch_suffix == "b2":
                scenario = "Economics & Politics Focus - Emphasis on trade warfare and political maneuvering"
            else:
                scenario = f"Branch {branch_suffix} development path"
            
            # Determine target ending for final stage branches
            target_ending = ""
            if "2030" in stage_config.get('period', ''):
                # Map final branches to specific endings
                ending_map = {
                    'foundation_b1_b1_b1_b1': ENDING_SCENARIOS[1]['name'],
                    'foundation_b1_b1_b1_b2': ENDING_SCENARIOS[2]['name'],
                    'foundation_b1_b1_b2_b1': ENDING_SCENARIOS[3]['name'],
                    'foundation_b1_b1_b2_b2': ENDING_SCENARIOS[4]['name'],
                    'foundation_b1_b2_b1_b1': ENDING_SCENARIOS[5]['name'],
                    'foundation_b1_b2_b1_b2': ENDING_SCENARIOS[6]['name'],
                    'foundation_b1_b2_b2_b1': ENDING_SCENARIOS[7]['name'],
                    'foundation_b1_b2_b2_b2': ENDING_SCENARIOS[8]['name'],
                }
                target_ending = ending_map.get(branch_path, "")
            
            all_tasks.append({
                'branch_path': branch_path,
                'parent_content': parent_content,
                'stage_config': stage_config,
                'scenario': scenario,
                'target_ending': target_ending
            })
    
    print(f"\n🌳 STAGE: {stage_name}")
    print("-" * 60)
    print(f"   Generating {len(all_tasks)} branches with conversations")
    
    # Execute branches sequentially (to avoid API rate limits)
    results = []
    for i, task in enumerate(all_tasks, 1):
        print(f"\n[{i}/{len(all_tasks)}] Processing branch: {task['branch_path']}")
        try:
            result = generate_story_segment_with_conversations(
                task['stage_config'],
                task['branch_path'],
                task['parent_content'],
                task['scenario'],
                task['target_ending']
            )
            results.append(result)
            print(f"✅ Branch complete: {result['word_count']:,} words")
        except Exception as e:
            print(f"❌ Branch failed: {e}")
            import traceback
            traceback.print_exc()
    
    return results

# Execute the complete generation process
try:
    total_start_time = time.time()
    all_segments = {}
    
    # Stage 1: Foundation (July 2025 - December 2026)
    stage_1_config = TIMELINE_STAGES["stage_1"]
    foundation_results = generate_parallel_branches_with_conversations(
        stage_1_config, None, "Foundation Period (July 2025 - December 2026)"
    )
    
    if foundation_results and len(foundation_results) > 0:
        foundation_result = foundation_results[0]
        all_segments["foundation"] = foundation_result
        print(f"✅ Foundation complete: {foundation_result['word_count']:,} words")
    else:
        print("❌ Foundation stage failed!")
        exit(1)
    
    # Stage 2: First Branching (January 2027 - December 2027) - 2 branches
    stage_2_config = TIMELINE_STAGES["stage_2"]
    stage_2_results = generate_parallel_branches_with_conversations(
        stage_2_config, {"foundation": foundation_result}, "First Branching (January 2027 - December 2027)"
    )
    
    # Convert to dictionary format and add to all_segments
    for result in stage_2_results:
        all_segments[result['branch_path']] = result
    
    print(f"✅ Stage 2 complete: {len(stage_2_results)} branches generated")
    stage_2_words = sum(r['word_count'] for r in stage_2_results)
    print(f"   Total Stage 2 words: {stage_2_words:,}")
    
    # Stage 3: Second Branching (January 2028 - December 2028) - 4 branches
    stage_2_segments = {result['branch_path']: result for result in stage_2_results}
    stage_3_config = TIMELINE_STAGES["stage_3"]
    stage_3_results = generate_parallel_branches_with_conversations(
        stage_3_config, stage_2_segments, "Second Branching (January 2028 - December 2028)"
    )
    
    # Add to all_segments
    for result in stage_3_results:
        all_segments[result['branch_path']] = result
    
    print(f"✅ Stage 3 complete: {len(stage_3_results)} branches generated")
    stage_3_words = sum(r['word_count'] for r in stage_3_results)
    print(f"   Total Stage 3 words: {stage_3_words:,}")
    
    # Stage 4: Third Branching (January 2029 - December 2029) - 8 branches
    stage_3_segments = {result['branch_path']: result for result in stage_3_results}
    stage_4_config = TIMELINE_STAGES["stage_4"]
    stage_4_results = generate_parallel_branches_with_conversations(
        stage_4_config, stage_3_segments, "Third Branching (January 2029 - December 2029)"
    )
    
    # Add to all_segments
    for result in stage_4_results:
        all_segments[result['branch_path']] = result
    
    print(f"✅ Stage 4 complete: {len(stage_4_results)} branches generated")
    stage_4_words = sum(r['word_count'] for r in stage_4_results)
    print(f"   Total Stage 4 words: {stage_4_words:,}")
    
    # Stage 5: Final Convergence to 8 Endings (January 2030 - December 2030) - 8 branches
    stage_4_segments = {result['branch_path']: result for result in stage_4_results}
    stage_5_config = TIMELINE_STAGES["stage_5"]
    stage_5_results = generate_parallel_branches_with_conversations(
        stage_5_config, stage_4_segments, "Final Convergence to 8 Endings (January 2030 - December 2030)"
    )
    
    # Add to all_segments
    for result in stage_5_results:
        all_segments[result['branch_path']] = result
    
    print(f"✅ Stage 5 complete: {len(stage_5_results)} branches generated")
    stage_5_words = sum(r['word_count'] for r in stage_5_results)
    print(f"   Total Stage 5 words: {stage_5_words:,}")
    
    # Calculate final statistics
    total_time = time.time() - total_start_time
    total_words = sum(segment['word_count'] for segment in all_segments.values())
    total_segments = len(all_segments)
    
    print("\n" + "=" * 80)
    print("🎉 COMPLETE 8-ENDING GENERATION FINISHED!")
    print("=" * 80)
    print(f"📊 Total Segments Generated: {total_segments}")
    print(f"📝 Total Words: {total_words:,}")
    print(f"⏱️  Total Time: {total_time/60:.1f} minutes")
    print(f"🎯 Target Achievement: {total_words/76000:.1%} of 76,000 word target")
    print(f"🌳 8-Ending System: Complete branching to all specified endings")
    
    # Show ending distribution
    print(f"\n🎯 ENDING DISTRIBUTION:")
    ending_count = 0
    for segment_id, segment_data in all_segments.items():
        if 'target_ending' in segment_data and segment_data['target_ending']:
            ending_count += 1
            print(f"   {segment_id}: {segment_data['target_ending']}")
    
    print(f"\n📈 Generation Summary:")
    print(f"   Stage 1 (Foundation): {foundation_result['word_count']:,} words (1 story)")
    print(f"   Stage 2 (2027): {stage_2_words:,} words ({len(stage_2_results)} branches)")
    print(f"   Stage 3 (2028): {stage_3_words:,} words ({len(stage_3_results)} branches)")
    print(f"   Stage 4 (2029): {stage_4_words:,} words ({len(stage_4_results)} branches)")
    print(f"   Stage 5 (2030): {stage_5_words:,} words ({len(stage_5_results)} branches)")
    print(f"   🎯 Endings reached: {ending_count}/8")
    
    # Export results
    print("\n📁 Creating comprehensive export...")
    export_dir = create_comprehensive_export(all_segments, generator, "enhanced_story_output_8_endings")
    print(f"✅ Export complete! Check directory: {export_dir}")
    
except Exception as e:
    print(f"❌ Generation failed: {e}")
    import traceback
    traceback.print_exc()
    print("Partial results may be available in the all_segments dictionary.")
    
    # Still try to export partial results
    if 'all_segments' in locals() and all_segments:
        print("📁 Attempting to export partial results...")
        try:
            export_dir = create_comprehensive_export(all_segments, generator, "enhanced_story_output_partial")
            print(f"✅ Partial export complete! Check directory: {export_dir}")
        except Exception as export_error:
            print(f"❌ Export also failed: {export_error}")

print("\n🏁 Enhanced 8-ending story generation process finished!")
print("Check the output directory for your complete branching narrative.")
print("🎯 8-Ending System: Each final branch leads to one of the specified endings.")

🚀 STARTING ENHANCED US-CHINA AGI ARMS RACE STORY GENERATION
✅ Available providers: google, openai, anthropic
✅ Enhanced world context ready: 35515 characters
🔧 Configuration:
   Provider: google
   Model: gemini-2.5-pro-preview-06-05
   Timeline: 5 stages
   8-Ending System: ✅ Active
   Total target words: 76,000
   Expected branches: 1 → 2 → 4 → 8 (leading to 8 specific endings)

❌ Failed to initialize generator: EnhancedStoryGenerator.__init__() got an unexpected keyword argument 'preferred_provider'
Please check your API keys and try again.

📖 STAGE: Foundation Period (July 2025 - December 2026)
------------------------------------------------------------

  💬 Generating: foundation
     Target: 20,000 words (6 conversations)
❌ Generation failed: 'EnhancedStoryGenerator' object has no attribute 'generate_stage_with_conversation'
Partial results may be available in the all_segments dictionary.

🏁 Enhanced 8-ending story generation process finished!
Check the output directory for your

Traceback (most recent call last):
  File "/var/folders/sh/z3hsvy7526b8xcjjphdq8hs00000gn/T/ipykernel_76443/1167826118.py", line 162, in <module>
    foundation_results = generate_parallel_branches_with_conversations(
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/sh/z3hsvy7526b8xcjjphdq8hs00000gn/T/ipykernel_76443/1167826118.py", line 87, in generate_parallel_branches_with_conversations
    return [generate_story_segment_with_conversations(stage_config, "foundation", "", "")]
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/sh/z3hsvy7526b8xcjjphdq8hs00000gn/T/ipykernel_76443/1167826118.py", line 52, in generate_story_segment_with_conversations
    segment = generator.generate_stage_with_conversation(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'EnhancedStoryGenerator' object has no attribute 'generate_stage_with_conversation'
